# Notebook 3 - Overdekking

Deze notebook berekent een brede doekoverdekking met meerdere identieke stangenmechanismen naast elkaar. De kinematica komt volledig uit `Notebook 1.ipynb`; de massa's van doek, voorbalk en beslag worden hier gekozen omdat ze afhangen van de breedte van de overdekking.

De dynamische aanpak blijft dezelfde als in Notebook 3: Newton-Euler per bewegende link, zwaartekracht, schuiverwrijving, pinwrijving, houdkracht, framebelasting en energiebalans. De resultaten worden opgeslagen als Notebook-4-loadcase zodat de motor, poelie, riem en rem nadien met dezelfde aandrijfnotebook kunnen worden gekozen.


## Setup en kinematica uit Notebook 1

De notebook leest alleen kinematische resultaten uit Notebook 1. Daardoor volgen trajecttijd, schuiverpositie, hoeken, snelheden en versnellingen automatisch mee wanneer de geometrie of beweging in Notebook 1 wordt aangepast.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from IPython.display import HTML

%matplotlib inline

kinematics_path = Path("notebook1_kinematica_results.npz")
if not kinematics_path.exists():
    raise FileNotFoundError("Run eerst 'Notebook 1.ipynb' zodat notebook1_kinematica_results.npz bestaat.")

kin_required = [
    "t", "Ts", "s", "ds", "dds",
    "theta3", "theta4", "theta5", "theta6", "theta7", "theta8",
    "dtheta3", "dtheta4", "dtheta5", "dtheta6", "dtheta7", "dtheta8",
    "ddtheta3", "ddtheta4", "ddtheta5", "ddtheta6", "ddtheta7", "ddtheta8",
    "Kx", "Ky", "Kdot_x", "Kdot_y", "Kdot_norm", "Kddot_x", "Kddot_y", "Kddot_norm",
    "cond", "residual_pos",
    "L1", "r3a", "r3b", "r4a", "r4b", "r5a", "r5b", "r6", "r7a", "r7b", "r8a", "r8b",
]
kin = np.load(kinematics_path)
missing_kin = [key for key in kin_required if key not in kin.files]
if missing_kin:
    raise KeyError(f"Ontbrekende keys in {kinematics_path}: {missing_kin}")

t = kin["t"]; Ts = float(kin["Ts"]); s = kin["s"]; ds = kin["ds"]; dds = kin["dds"]
n_steps = len(t); T_cycle = float(t[-1] - t[0])
theta3 = kin["theta3"]; theta4 = kin["theta4"]; theta5 = kin["theta5"]
theta6 = kin["theta6"]; theta7 = kin["theta7"]; theta8 = kin["theta8"]
dtheta3 = kin["dtheta3"]; dtheta4 = kin["dtheta4"]; dtheta5 = kin["dtheta5"]
dtheta6 = kin["dtheta6"]; dtheta7 = kin["dtheta7"]; dtheta8 = kin["dtheta8"]
ddtheta3 = kin["ddtheta3"]; ddtheta4 = kin["ddtheta4"]; ddtheta5 = kin["ddtheta5"]
ddtheta6 = kin["ddtheta6"]; ddtheta7 = kin["ddtheta7"]; ddtheta8 = kin["ddtheta8"]
Kx = kin["Kx"]; Ky = kin["Ky"]; Kdot_x = kin["Kdot_x"]; Kdot_y = kin["Kdot_y"]
Kdot_norm = kin["Kdot_norm"]; Kddot_x = kin["Kddot_x"]; Kddot_y = kin["Kddot_y"]; Kddot_norm = kin["Kddot_norm"]
cond = kin["cond"]; residual_pos = kin["residual_pos"]
L1 = float(kin["L1"]); r3a = float(kin["r3a"]); r3b = float(kin["r3b"])
r4a = float(kin["r4a"]); r4b = float(kin["r4b"]); r5a = float(kin["r5a"]); r5b = float(kin["r5b"])
r6 = float(kin["r6"]); r7a = float(kin["r7a"]); r7b = float(kin["r7b"]); r8a = float(kin["r8a"]); r8b = float(kin["r8b"])
L2 = 0.0; L3 = r3a + r3b; L4 = r4a + r4b; L5 = r5a + r5b; L6 = r6; L7 = r7a + r7b; L8 = r8a + r8b

unknown_labels = [
    "R_Ax", "F_act_y", "M_A", "C_x", "C_y", "B_x", "B_y", "D_x", "D_y", "E_x", "E_y",
    "F_x", "F_y", "G_x", "G_y", "H_x", "H_y", "I_x", "I_y", "J_x", "J_y",
]
unknown_index = {name: i for i, name in enumerate(unknown_labels)}
n_unknowns = len(unknown_labels)

s_open = float(np.min(s))
s_closed = float(np.max(s))
stroke = s_closed - s_open
canopy_depth = float(np.max(Kx))

print("Kinematica ingeladen uit Notebook 1:")
print(kinematics_path.resolve())
print(f"aantal tijdstappen                 : {n_steps}")
print(f"simulatieduur                      : {T_cycle:.3f} s")
print(f"s_open / s_closed                  : {s_open:.4f} m / {s_closed:.4f} m")
print(f"geschatte uitval op basis van K    : {canopy_depth:.3f} m")
print(f"max cond(A) Notebook 1             : {np.max(cond):.2f}")
print(f"max sluitingsfout Notebook 1       : {np.max(residual_pos):.3e} m")


## Overdekkingsparameters en massamodel

De brede overdekking gebruikt dezelfde kinematica per mechanisme, maar de last in K wordt opnieuw bepaald. De voorbalk ligt tussen de K-punten en het doek wordt als oppervlaktebelasting gekozen. Voor de motorbelasting is het conservatief om een instelbaar deel van de doekmassa als equivalente K-last mee te nemen.

De eerste balkcontrole hieronder blijft een zwaartekrachtcontrole op doorbuiging. Daarna volgt een aparte structurele weercontrole voor wind, sneeuw en regenwater. Die extra controle is een instelbare eerste dimensionering, geen gecertificeerde Eurocode-eindberekening.

Voor meer dan twee mechanismen blijft dit een identiek-mechanisme-model: de totale last wordt gelijkmatig over de mechanismen verdeeld. Dat is geschikt voor een eerste motorvergelijking, maar bij een doorlopende voorbalk kan een middensteun lokaal meer opnemen dan de eindsteunen. De balkgrafiek toont daarom vooral of een extra steun zinvol is; lokale steunreacties van een doorlopende balk blijven een detailcontrole.


In [ ]:
# ============================================================
# Instelbare overdekkingsparameters
# ============================================================

canopy_width = 6.0
mechanism_count_total = 2
fabric_areal_density = 0.35
fabric_mass_to_K_fraction = 1.00
fittings_mass_per_K = 1.50

front_beam_profile = "200x100x5"
aluminium_density = 2700.0
aluminium_E = 69e9
custom_beam_height = 0.200
custom_beam_width = 0.100
custom_beam_thickness = 0.005

g = 9.81
beam_profile_catalog = {
    "80x40x3":   dict(height=0.080, width=0.040, thickness=0.003),
    "100x50x3":  dict(height=0.100, width=0.050, thickness=0.003),
    "120x60x4":  dict(height=0.120, width=0.060, thickness=0.004),
    "140x60x4":  dict(height=0.140, width=0.060, thickness=0.004),
    "160x80x5":  dict(height=0.160, width=0.080, thickness=0.005),
    "180x80x5":  dict(height=0.180, width=0.080, thickness=0.005),
    "200x100x5": dict(height=0.200, width=0.100, thickness=0.005),
    "200x100x6": dict(height=0.200, width=0.100, thickness=0.006),
    "220x100x6": dict(height=0.220, width=0.100, thickness=0.006),
}

if mechanism_count_total < 2:
    raise ValueError("Voor een brede overdekking zijn minstens twee mechanismen nodig.")
if canopy_width <= 0 or canopy_depth <= 0:
    raise ValueError("canopy_width en canopy_depth moeten positief zijn.")
if not (0.0 <= fabric_mass_to_K_fraction <= 1.0):
    raise ValueError("fabric_mass_to_K_fraction moet tussen 0 en 1 liggen.")

if front_beam_profile == "custom":
    beam_height = float(custom_beam_height)
    beam_width = float(custom_beam_width)
    beam_thickness = float(custom_beam_thickness)
elif front_beam_profile in beam_profile_catalog:
    profile = beam_profile_catalog[front_beam_profile]
    beam_height = profile["height"]
    beam_width = profile["width"]
    beam_thickness = profile["thickness"]
else:
    raise ValueError(f"Onbekend front_beam_profile: {front_beam_profile}")

if beam_thickness <= 0 or 2 * beam_thickness >= min(beam_height, beam_width):
    raise ValueError("Ongeldige kokergeometrie voor de voorbalk.")

def rectangular_tube_area(width, height, thickness):
    return width * height - (width - 2 * thickness) * (height - 2 * thickness)

def rectangular_tube_I_vertical(width, height, thickness):
    return (width * height**3 - (width - 2 * thickness) * (height - 2 * thickness)**3) / 12.0

beam_area = rectangular_tube_area(beam_width, beam_height, beam_thickness)
beam_I = rectangular_tube_I_vertical(beam_width, beam_height, beam_thickness)
front_beam_mass_per_m = beam_area * aluminium_density
front_beam_mass_total = front_beam_mass_per_m * canopy_width
canopy_area = canopy_width * canopy_depth
fabric_mass_total = canopy_area * fabric_areal_density
mechanism_spacing = canopy_width / (mechanism_count_total - 1)
beam_span = mechanism_spacing
support_z_positions = np.linspace(-canopy_width / 2.0, canopy_width / 2.0, mechanism_count_total)

front_beam_mass_per_K = front_beam_mass_total / mechanism_count_total
fabric_mass_per_K = fabric_mass_to_K_fraction * fabric_mass_total / mechanism_count_total
payload_mass_K_equivalent = front_beam_mass_per_K + fabric_mass_per_K + fittings_mass_per_K
payload_mass_K_half_fabric = front_beam_mass_per_K + 0.50 * fabric_mass_total / mechanism_count_total + fittings_mass_per_K

slider_mass = 1.50
rod_outer_diameter = 0.030
rod_wall_thickness = 0.002
rod_material_density = aluminium_density
rod_fittings_line_mass_allowance = 0.075
rod_inner_diameter = rod_outer_diameter - 2.0 * rod_wall_thickness
if rod_inner_diameter <= 0:
    raise ValueError("rod_wall_thickness is te groot voor rod_outer_diameter.")
rod_tube_area = np.pi / 4.0 * (rod_outer_diameter**2 - rod_inner_diameter**2)
rod_tube_mass_per_m = rod_tube_area * rod_material_density
line_mass_density = rod_tube_mass_per_m + rod_fittings_line_mass_allowance
mass_scale = 1.0
inertia_scale = 1.0
masses = {
    2: slider_mass * mass_scale,
    3: line_mass_density * L3 * mass_scale,
    4: line_mass_density * L4 * mass_scale,
    5: line_mass_density * L5 * mass_scale,
    6: line_mass_density * L6 * mass_scale,
    7: line_mass_density * L7 * mass_scale,
    8: line_mass_density * L8 * mass_scale,
}
inertias = {
    2: 0.0,
    3: masses[3] * L3**2 / 12.0 * inertia_scale,
    4: masses[4] * L4**2 / 12.0 * inertia_scale,
    5: masses[5] * L5**2 / 12.0 * inertia_scale,
    6: masses[6] * L6**2 / 12.0 * inertia_scale,
    7: masses[7] * L7**2 / 12.0 * inertia_scale,
    8: masses[8] * L8**2 / 12.0 * inertia_scale,
}
payload_mass_K = payload_mass_K_equivalent
moving_link_mass = sum(masses.values())
total_model_mass = moving_link_mass + payload_mass_K
total_system_model_mass = mechanism_count_total * total_model_mass

def beam_deflection_for(width, count, I, mass_per_m, fabric_density, depth, fabric_fraction):
    span = width / (count - 1)
    q_mass = mass_per_m + fabric_density * depth * fabric_fraction
    q_force = q_mass * g
    return 5.0 * q_force * span**4 / (384.0 * aluminium_E * I)

beam_line_load_mass = front_beam_mass_per_m + fabric_areal_density * canopy_depth * fabric_mass_to_K_fraction
beam_line_load_force = beam_line_load_mass * g
beam_deflection_max = beam_deflection_for(
    canopy_width, mechanism_count_total, beam_I, front_beam_mass_per_m,
    fabric_areal_density, canopy_depth, fabric_mass_to_K_fraction,
)
beam_deflection_limit_L300 = beam_span / 300.0
beam_deflection_ratio = beam_deflection_max / beam_span

print("Overdekkingsmodel:")
print(f"breedte / uitval                  : {canopy_width:.2f} m / {canopy_depth:.2f} m")
print(f"aantal mechanismen                : {mechanism_count_total}")
print(f"mechanisme-afstand / balkspan     : {mechanism_spacing:.2f} m / {beam_span:.2f} m")
print(f"voorbalkprofiel                   : {front_beam_profile}, m' = {front_beam_mass_per_m:.2f} kg/m")
print(f"voorbalkmassa totaal              : {front_beam_mass_total:.2f} kg")
print(f"stangprofiel                      : aluminium buis {1000*rod_outer_diameter:.0f}x{1000*rod_wall_thickness:.1f} mm + {rod_fittings_line_mass_allowance:.3f} kg/m beslag")
print(f"lijnmassa stangen                 : {line_mass_density:.4f} kg/m")
print(f"doekmassa totaal                  : {fabric_mass_total:.2f} kg")
print(f"payload_mass_K equivalent         : {payload_mass_K:.2f} kg per mechanisme")
print(f"payload_mass_K bij 50% doek naar K: {payload_mass_K_half_fabric:.2f} kg per mechanisme")
print(f"totale modelmassa per mechanisme  : {total_model_mass:.2f} kg")
print(f"totale modelmassa alle mechanismen: {total_system_model_mass:.2f} kg")
print(f"voorbalkdoorbuiging zwaartekracht : {1000*beam_deflection_max:.1f} mm ({beam_deflection_ratio:.4f} van span)")
print(f"richtwaarde L/300                 : {1000*beam_deflection_limit_L300:.1f} mm")

fig_mass, ax_mass = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
fig_mass.suptitle("Overdekkingsmassa en voorbalkcontrole")
ax_mass[0].bar(["voorbalk", "doek", "K per mech"], [front_beam_mass_total, fabric_mass_total, payload_mass_K])
ax_mass[0].set_ylabel("massa [kg]")
ax_mass[0].set_title("Massa-opbouw")
ax_mass[0].grid(True, axis="y")

width_grid = np.linspace(4.0, 8.0, 81)
defl_2 = np.array([beam_deflection_for(w, 2, beam_I, front_beam_mass_per_m, fabric_areal_density, canopy_depth, fabric_mass_to_K_fraction) for w in width_grid])
defl_3 = np.array([beam_deflection_for(w, 3, beam_I, front_beam_mass_per_m, fabric_areal_density, canopy_depth, fabric_mass_to_K_fraction) for w in width_grid])
ax_mass[1].plot(width_grid, 1000 * defl_2, label="2 mechanismen")
ax_mass[1].plot(width_grid, 1000 * defl_3, label="3 mechanismen")
ax_mass[1].axvline(canopy_width, color="black", ls=":", label="gekozen breedte")
ax_mass[1].set_xlabel("breedte [m]"); ax_mass[1].set_ylabel("doorbuiging [mm]")
ax_mass[1].set_title("Effect van breedte")
ax_mass[1].grid(True); ax_mass[1].legend()

profile_names = list(beam_profile_catalog.keys())
profile_deflections = []
for name in profile_names:
    p = beam_profile_catalog[name]
    A_i = rectangular_tube_area(p["width"], p["height"], p["thickness"])
    I_i = rectangular_tube_I_vertical(p["width"], p["height"], p["thickness"])
    m_i = A_i * aluminium_density
    profile_deflections.append(1000 * beam_deflection_for(canopy_width, mechanism_count_total, I_i, m_i, fabric_areal_density, canopy_depth, fabric_mass_to_K_fraction))
ax_mass[2].bar(profile_names, profile_deflections)
ax_mass[2].set_ylabel("doorbuiging [mm]")
ax_mass[2].set_title("Profielvergelijking")
ax_mass[2].tick_params(axis="x", rotation=25)
ax_mass[2].grid(True, axis="y")
plt.show()


## Structurele controle voorbalk

De voorbalk tussen de K-punten wordt hier als eenvoudig opgelegde rechthoekige aluminium koker per overspanning benaderd. Deze controle gebruikt dezelfde `canopy_width`, `mechanism_count_total`, `canopy_depth` en profielkeuze als het massamodel. Naast eigengewicht worden regenwater, sneeuw en winddruk/-zuiging als instelbare oppervlaktebelastingen omgerekend naar een lijnlast op de voorbalk.

De berekening is bedoeld als eerste ontwerpcontrole: ze controleert buiging, dwarskracht, torsie, doorbuiging en twist, maar vervangt geen volledige normberekening met gebouwlocatie, randvoorwaarden, windrichting, doekspanning, wateraccumulatie en bevestigingsdetails.


In [ ]:
# ============================================================
# Structurele weercontrole voor de voorbalk
# ============================================================

enable_weather_structural_check = True

deflection_limit_ratio = 300.0
deflection_limit_abs = 0.020
twist_limit_deg = 2.0
aluminium_yield_strength = 150e6
stress_safety_factor = 1.5
aluminium_nu = 0.33

wind_basic_velocity = 26.0
wind_peak_pressure_factor = 1.8
wind_cp_down = 0.8
wind_cp_uplift = -1.2
rho_air = 1.25

ground_snow_load = 0.50e3
snow_shape_coefficient = 0.8
rain_water_depth_cases_mm = [0, 10, 25]
rho_water = 1000.0

front_beam_tributary_depth_fraction = 0.50
front_beam_load_eccentricity = 0.04
include_manual_extreme_load_case = False
manual_extreme_area_pressure = 1.00e3
manual_extreme_case_name = "handmatig extreem"

if deflection_limit_ratio <= 0 or deflection_limit_abs <= 0:
    raise ValueError("Doorbuigingslimieten moeten positief zijn.")
if twist_limit_deg <= 0 or aluminium_yield_strength <= 0 or stress_safety_factor <= 0:
    raise ValueError("Sterkte-, twist- en veiligheidsparameters moeten positief zijn.")
if not (0.0 < front_beam_tributary_depth_fraction <= 1.0):
    raise ValueError("front_beam_tributary_depth_fraction moet tussen 0 en 1 liggen.")
if front_beam_load_eccentricity < 0:
    raise ValueError("front_beam_load_eccentricity mag niet negatief zijn.")


def rectangular_tube_I_weak(width, height, thickness):
    return (height * width**3 - (height - 2 * thickness) * (width - 2 * thickness)**3) / 12.0


def rectangular_tube_torsion_constant(width, height, thickness):
    # Bredt-Batho benadering voor een dunwandige gesloten rechthoekige koker.
    width_mid = width - thickness
    height_mid = height - thickness
    enclosed_area_mid = width_mid * height_mid
    wall_sum = 2.0 * width_mid / thickness + 2.0 * height_mid / thickness
    return 4.0 * enclosed_area_mid**2 / wall_sum


beam_I_strong = beam_I
beam_I_weak = rectangular_tube_I_weak(beam_width, beam_height, beam_thickness)
beam_section_modulus_strong = beam_I_strong / (0.5 * beam_height)
beam_section_modulus_weak = beam_I_weak / (0.5 * beam_width)
beam_shear_area = 2.0 * beam_thickness * beam_height
beam_torsion_constant = rectangular_tube_torsion_constant(beam_width, beam_height, beam_thickness)
beam_G = aluminium_E / (2.0 * (1.0 + aluminium_nu))
beam_torsion_enclosed_area_mid = (beam_width - beam_thickness) * (beam_height - beam_thickness)

beam_self_line_load = front_beam_mass_per_m * g
fabric_self_line_load = fabric_areal_density * canopy_depth * front_beam_tributary_depth_fraction * g
wind_basic_pressure = 0.5 * rho_air * wind_basic_velocity**2
wind_peak_pressure = wind_peak_pressure_factor * wind_basic_pressure
wind_down_pressure = wind_peak_pressure * wind_cp_down
wind_uplift_pressure = wind_peak_pressure * wind_cp_uplift
snow_pressure = ground_snow_load * snow_shape_coefficient
rain_pressure_cases = rho_water * g * np.asarray(rain_water_depth_cases_mm, dtype=float) / 1000.0

allowable_deflection = min(beam_span / deflection_limit_ratio, deflection_limit_abs)
allowable_stress = aluminium_yield_strength / stress_safety_factor
allowable_twist_rad = np.deg2rad(twist_limit_deg)


def evaluate_beam_weather_case(name, area_pressure):
    area_pressure = float(area_pressure)
    tributary_depth = canopy_depth * front_beam_tributary_depth_fraction
    weather_line_load = area_pressure * tributary_depth
    q_line = beam_self_line_load + fabric_self_line_load + weather_line_load
    q_eccentric = fabric_self_line_load + weather_line_load

    V_max = q_line * beam_span / 2.0
    M_max = q_line * beam_span**2 / 8.0
    delta_max = 5.0 * q_line * beam_span**4 / (384.0 * aluminium_E * beam_I_strong)
    sigma_bending = abs(M_max) / beam_section_modulus_strong
    tau_shear = abs(V_max) / beam_shear_area

    torsion_line_moment = q_eccentric * front_beam_load_eccentricity
    T_max = torsion_line_moment * beam_span / 2.0
    twist_max = torsion_line_moment * beam_span**2 / (8.0 * beam_G * beam_torsion_constant)
    tau_torsion = abs(T_max) / (2.0 * beam_torsion_enclosed_area_mid * beam_thickness)

    tau_equivalent = np.sqrt(tau_shear**2 + tau_torsion**2)
    von_mises = np.sqrt(sigma_bending**2 + 3.0 * tau_equivalent**2)
    util_deflection = abs(delta_max) / allowable_deflection
    util_stress = von_mises / allowable_stress
    util_torsion = abs(twist_max) / allowable_twist_rad

    return dict(
        name=name,
        area_pressure=area_pressure,
        q_line=q_line,
        q_eccentric=q_eccentric,
        V_max=V_max,
        M_max=M_max,
        delta_max=delta_max,
        sigma_bending=sigma_bending,
        tau_shear=tau_shear,
        T_max=T_max,
        twist_max=twist_max,
        tau_torsion=tau_torsion,
        von_mises=von_mises,
        util_deflection=util_deflection,
        util_stress=util_stress,
        util_torsion=util_torsion,
    )


if enable_weather_structural_check:
    beam_weather_cases = [evaluate_beam_weather_case("eigengewicht + doek", 0.0)]
    for rain_mm, rain_pressure in zip(rain_water_depth_cases_mm, rain_pressure_cases):
        if float(rain_mm) > 0.0:
            beam_weather_cases.append(evaluate_beam_weather_case(f"regenwater {rain_mm:g} mm", rain_pressure))
    beam_weather_cases.append(evaluate_beam_weather_case("sneeuw", snow_pressure))
    beam_weather_cases.append(evaluate_beam_weather_case("wind neerwaarts", wind_down_pressure))
    beam_weather_cases.append(evaluate_beam_weather_case("wind uplift", wind_uplift_pressure))
    if include_manual_extreme_load_case:
        beam_weather_cases.append(evaluate_beam_weather_case(manual_extreme_case_name, manual_extreme_area_pressure))
else:
    beam_weather_cases = [evaluate_beam_weather_case("uitgeschakeld", 0.0)]
    for key in ["q_line", "q_eccentric", "V_max", "M_max", "delta_max", "sigma_bending", "tau_shear", "T_max", "twist_max", "tau_torsion", "von_mises", "util_deflection", "util_stress", "util_torsion"]:
        beam_weather_cases[0][key] = 0.0

weather_case_names = np.array([case["name"] for case in beam_weather_cases], dtype=str)
weather_area_pressure_cases = np.array([case["area_pressure"] for case in beam_weather_cases], dtype=float)
beam_q_line_cases = np.array([case["q_line"] for case in beam_weather_cases], dtype=float)
beam_V_max_cases = np.array([case["V_max"] for case in beam_weather_cases], dtype=float)
beam_M_max_cases = np.array([case["M_max"] for case in beam_weather_cases], dtype=float)
beam_deflection_cases = np.array([case["delta_max"] for case in beam_weather_cases], dtype=float)
beam_T_max_cases = np.array([case["T_max"] for case in beam_weather_cases], dtype=float)
beam_twist_cases = np.array([case["twist_max"] for case in beam_weather_cases], dtype=float)
beam_sigma_bending_cases = np.array([case["sigma_bending"] for case in beam_weather_cases], dtype=float)
beam_tau_shear_cases = np.array([case["tau_shear"] for case in beam_weather_cases], dtype=float)
beam_tau_torsion_cases = np.array([case["tau_torsion"] for case in beam_weather_cases], dtype=float)
beam_von_mises_cases = np.array([case["von_mises"] for case in beam_weather_cases], dtype=float)
beam_utilization_deflection = np.array([case["util_deflection"] for case in beam_weather_cases], dtype=float)
beam_utilization_stress = np.array([case["util_stress"] for case in beam_weather_cases], dtype=float)
beam_utilization_torsion = np.array([case["util_torsion"] for case in beam_weather_cases], dtype=float)
beam_utilization_max = np.maximum.reduce([beam_utilization_deflection, beam_utilization_stress, beam_utilization_torsion])
beam_governing_index = int(np.argmax(beam_utilization_max))
beam_governing_case = str(weather_case_names[beam_governing_index])
beam_structural_ok = bool(np.max(beam_utilization_max) <= 1.0)

if not enable_weather_structural_check:
    beam_structural_status = "controle uitgeschakeld"
elif np.max(beam_utilization_max) <= 1.0:
    beam_structural_status = "acceptabel binnen deze aannames"
elif np.max(beam_utilization_max) <= 1.25:
    beam_structural_status = "twijfelachtig; profiel of steunafstand herbekijken"
else:
    beam_structural_status = "niet gebruiken in deze weersomstandigheden"

print("Structurele controle voorbalk:")
print(f"wind piekdruk q_p                    : {wind_peak_pressure:.0f} Pa")
print(f"wind neerwaarts / uplift             : {wind_down_pressure:.0f} Pa / {wind_uplift_pressure:.0f} Pa")
print(f"sneeuwdruk                           : {snow_pressure:.0f} Pa")
print(f"tributaire doekdiepte voorbalk        : {front_beam_tributary_depth_fraction*canopy_depth:.2f} m")
print(f"toelaatbare doorbuiging               : {1000*allowable_deflection:.1f} mm")
print(f"toelaatbare spanning incl. veiligheid : {allowable_stress/1e6:.1f} MPa")
print(f"maatgevende loadcase                  : {beam_governing_case}")
print(f"max doorbuiging                       : {1000*abs(beam_deflection_cases[beam_governing_index]):.1f} mm")
print(f"max von Mises-spanning                : {beam_von_mises_cases[beam_governing_index]/1e6:.1f} MPa")
print(f"max torsiehoek                        : {np.rad2deg(abs(beam_twist_cases[beam_governing_index])):.2f} deg")
print(f"conclusie                             : {beam_structural_status}")

print("\nLoadcase-overzicht:")
print("case                         q_line[N/m]   delta[mm]   sigma_vm[MPa]   util_max")
for i, name in enumerate(weather_case_names):
    print(f"{name:<28s} {beam_q_line_cases[i]:>10.1f} {1000*beam_deflection_cases[i]:>11.1f} {beam_von_mises_cases[i]/1e6:>15.1f} {beam_utilization_max[i]:>10.2f}")


def evaluate_profile_weather_screen(profile_width, profile_height, profile_thickness):
    A_i = rectangular_tube_area(profile_width, profile_height, profile_thickness)
    I_i = rectangular_tube_I_vertical(profile_width, profile_height, profile_thickness)
    S_i = I_i / (0.5 * profile_height)
    shear_area_i = 2.0 * profile_thickness * profile_height
    J_i = rectangular_tube_torsion_constant(profile_width, profile_height, profile_thickness)
    enclosed_area_i = (profile_width - profile_thickness) * (profile_height - profile_thickness)
    mass_per_m_i = A_i * aluminium_density
    q_self_i = mass_per_m_i * g + fabric_self_line_load

    util_cases = []
    deflection_cases_i = []
    stress_cases_i = []
    twist_cases_i = []
    for area_pressure in weather_area_pressure_cases:
        weather_line_load = float(area_pressure) * canopy_depth * front_beam_tributary_depth_fraction
        q_line_i = q_self_i + weather_line_load
        q_ecc_i = fabric_self_line_load + weather_line_load
        V_i = q_line_i * beam_span / 2.0
        M_i = q_line_i * beam_span**2 / 8.0
        delta_i = 5.0 * q_line_i * beam_span**4 / (384.0 * aluminium_E * I_i)
        sigma_i = abs(M_i) / S_i
        tau_i = abs(V_i) / shear_area_i
        T_i = q_ecc_i * front_beam_load_eccentricity * beam_span / 2.0
        twist_i = q_ecc_i * front_beam_load_eccentricity * beam_span**2 / (8.0 * beam_G * J_i)
        tau_t_i = abs(T_i) / (2.0 * enclosed_area_i * profile_thickness)
        von_mises_i = np.sqrt(sigma_i**2 + 3.0 * (tau_i**2 + tau_t_i**2))
        util_i = max(abs(delta_i) / allowable_deflection, von_mises_i / allowable_stress, abs(twist_i) / allowable_twist_rad)
        util_cases.append(util_i)
        deflection_cases_i.append(delta_i)
        stress_cases_i.append(von_mises_i)
        twist_cases_i.append(twist_i)

    util_cases = np.asarray(util_cases)
    governing_i = int(np.argmax(util_cases))
    beam_mass_total_i = mass_per_m_i * canopy_width
    payload_mass_K_i = beam_mass_total_i / mechanism_count_total + fabric_mass_per_K + fittings_mass_per_K
    return dict(
        mass_per_m=mass_per_m_i,
        payload_mass_K=payload_mass_K_i,
        max_util=float(util_cases[governing_i]),
        governing_case=str(weather_case_names[governing_i]),
        max_abs_deflection=float(np.max(np.abs(deflection_cases_i))),
        max_von_mises=float(np.max(stress_cases_i)),
        max_abs_twist=float(np.max(np.abs(twist_cases_i))),
    )

profile_screen_names = np.array(list(beam_profile_catalog.keys()), dtype=str)
profile_screen_mass_per_m = np.zeros(len(profile_screen_names))
profile_screen_payload_mass_K = np.zeros(len(profile_screen_names))
profile_screen_max_util = np.zeros(len(profile_screen_names))
profile_screen_governing_case = np.empty(len(profile_screen_names), dtype=object)
profile_screen_max_deflection = np.zeros(len(profile_screen_names))
profile_screen_max_von_mises = np.zeros(len(profile_screen_names))
profile_screen_max_twist = np.zeros(len(profile_screen_names))

for i, profile_name in enumerate(profile_screen_names):
    profile_i = beam_profile_catalog[str(profile_name)]
    screen = evaluate_profile_weather_screen(profile_i["width"], profile_i["height"], profile_i["thickness"])
    profile_screen_mass_per_m[i] = screen["mass_per_m"]
    profile_screen_payload_mass_K[i] = screen["payload_mass_K"]
    profile_screen_max_util[i] = screen["max_util"]
    profile_screen_governing_case[i] = screen["governing_case"]
    profile_screen_max_deflection[i] = screen["max_abs_deflection"]
    profile_screen_max_von_mises[i] = screen["max_von_mises"]
    profile_screen_max_twist[i] = screen["max_abs_twist"]

profile_screen_ok = profile_screen_max_util <= 1.0
print("\nProfielscreening bij gekozen breedte en weerparameters:")
print("profiel        kg/m   payloadK   util_max   governing       ok")
for i, profile_name in enumerate(profile_screen_names):
    ok_label = "ja" if profile_screen_ok[i] else "nee"
    print(f"{profile_name:<12s} {profile_screen_mass_per_m[i]:>5.2f} {profile_screen_payload_mass_K[i]:>9.2f} {profile_screen_max_util[i]:>9.2f}   {profile_screen_governing_case[i]:<13s} {ok_label}")

if enable_weather_structural_check:
    fig_weather, ax_weather = plt.subplots(figsize=(11, 4.5), constrained_layout=True)
    x = np.arange(len(weather_case_names))
    width_bar = 0.25
    ax_weather.bar(x - width_bar, beam_utilization_deflection, width_bar, label="doorbuiging")
    ax_weather.bar(x, beam_utilization_stress, width_bar, label="spanning")
    ax_weather.bar(x + width_bar, beam_utilization_torsion, width_bar, label="torsie")
    ax_weather.axhline(1.0, color="black", linewidth=1.0, linestyle="--", label="limiet")
    ax_weather.set_xticks(x)
    ax_weather.set_xticklabels(weather_case_names, rotation=25, ha="right")
    ax_weather.set_ylabel("benuttingsgraad [-]")
    ax_weather.set_title("Voorbalk: benutting per weer-loadcase")
    ax_weather.grid(True, axis="y")
    ax_weather.legend(ncol=4)
    plt.show()

    x_beam = np.linspace(0.0, beam_span, 250)
    q_gov = beam_q_line_cases[beam_governing_index]
    R_gov = q_gov * beam_span / 2.0
    V_curve = R_gov - q_gov * x_beam
    M_curve = R_gov * x_beam - 0.5 * q_gov * x_beam**2
    delta_curve = q_gov * x_beam * (beam_span**3 - 2.0 * beam_span * x_beam**2 + x_beam**3) / (24.0 * aluminium_E * beam_I_strong)

    fig_diag, ax_diag = plt.subplots(3, 1, figsize=(10, 7), sharex=True, constrained_layout=True)
    fig_diag.suptitle(f"Voorbalkdiagrammen - {beam_governing_case}")
    ax_diag[0].plot(x_beam, V_curve)
    ax_diag[0].axhline(0.0, color="black", linewidth=0.8)
    ax_diag[0].set_ylabel("V [N]")
    ax_diag[0].grid(True)
    ax_diag[1].plot(x_beam, M_curve)
    ax_diag[1].axhline(0.0, color="black", linewidth=0.8)
    ax_diag[1].set_ylabel("M [Nm]")
    ax_diag[1].grid(True)
    ax_diag[2].plot(x_beam, 1000.0 * delta_curve)
    ax_diag[2].axhline(1000.0 * allowable_deflection, color="tab:red", linestyle="--", linewidth=0.9, label="limiet")
    ax_diag[2].axhline(-1000.0 * allowable_deflection, color="tab:red", linestyle="--", linewidth=0.9)
    ax_diag[2].set_xlabel("positie over balkspan [m]")
    ax_diag[2].set_ylabel("delta [mm]")
    ax_diag[2].grid(True)
    ax_diag[2].legend()
    plt.show()


## Belastingsparameters

De wrijvingsparameters volgen dezelfde betekenis als in Notebook 3. De trekveer wordt optioneel als bekende verticale kracht op de schuiver toegevoegd. De baseline zonder veer blijft de hoofdcase; de veer-case wordt apart opgeslagen zodat de aandrijving rechtstreeks kan worden vergeleken.


In [ ]:
with_gravity = True
g = 9.81
g_vec = np.array([0.0, -g])

# Schuivergeleiding. Voor een outdoor collar met rollen/glijblokken is de effectieve
# wrijving typisch lager dan droge metaal-op-metaal, maar hoger dan ideale kogellagers.
# Bronorde: droge kunststof/staal glijgeleidingen ca. 0.05-0.23; daarom kiezen we
# dynamisch 0.08 en statisch 0.12 als realistische, conservatieve basis.
mu_slider = 0.08
mu_slider_static = 0.12
c_slider = 0.0
v_eps = 1e-3

# Scharnier-/buswrijving. Eenvoudige Coulomb-benadering met pinradius.
# mu_pin = 0.05 is een orde-waarde voor gesmeerde of lage-wrijving bus-/penscharniertjes.
include_pin_friction = True
mu_pin = 0.05
pin_radius = 0.006
omega_eps = 1e-3
# 20 iteraties zijn voldoende om de tanh-geregulariseerde Coulombwrijving te convergeren.
# Met 3 iteraties bedroeg de maximale iteratieverandering ~4e-3 (ruim boven de tolerantie).
# Met 20 iteraties convergeert het stelsel typisch tot onder 1e-8 binnen 8-12 stappen.
friction_iterations = 20
friction_tol = 1e-8

actuator_efficiency = 0.78  # tandriem + reductiekast, zie NB4
actuator_safety_factor = 1.50
drive_pulley_radius_nb3 = 0.025  # [m] referentiepoelie voor quick check; Notebook 4 kiest definitief
drive_travel_per_rev_nb3 = 2 * np.pi * drive_pulley_radius_nb3  # [m/omw] riemverplaatsing per poelie-omwenteling
pulley_radius_nb3 = drive_pulley_radius_nb3  # compatibility alias voor oudere outputs/tekst
screw_lead = drive_travel_per_rev_nb3  # compatibility alias; geen schroefspindel in dit ontwerp

print("Belastingsparameters:")
print(f"g = {g:.2f} m/s^2")
print(f"mu_slider = {mu_slider:.3f}, mu_slider_static = {mu_slider_static:.3f}, c_slider = {c_slider:.2f} N s/m")
print(f"mu_pin = {mu_pin:.3f}, pin_radius = {pin_radius*1000:.1f} mm, scharnierwrijving actief = {include_pin_friction}")
print(f"friction_iterations = {friction_iterations}, friction_tol = {friction_tol:.0e}")
print(f"referentiepoelieradius = {drive_pulley_radius_nb3*1000:.1f} mm, actuator_efficiency = {actuator_efficiency:.2f}")

# Optionele trekveer per mechanisme. De baseline zonder veer blijft de hoofdcase.
compute_spring_assist_case = True
use_spring_assist_for_main_output = False
spring_count_per_mechanism = 2
spring_design_mode = "fraction_of_baseline_hold"  # "fraction_of_baseline_hold" of "manual"
spring_assist_fraction_open = 0.60
spring_assist_fraction_closed = 0.70
spring_max_assist_fraction = 0.80
spring_force_open_total_manual = 40.0
spring_force_closed_total_manual = 120.0

# Directe trekveren per mechanisme. Geen kabel-/poelieverhouding: de veer volgt
# dezelfde slag als de schuiver. Lage veerconstante omdat de schuiverslag groot is.
spring_direct_rate_per_spring_N_per_mm = 0.010  # [N/mm] per trekveer
spring_physical_rate_per_spring_N_per_mm = spring_direct_rate_per_spring_N_per_mm  # compatibility alias
spring_motion_ratio = 1.0  # directe montage: dx_veer = ds_schuiver

if spring_count_per_mechanism < 1:
    raise ValueError("spring_count_per_mechanism moet minstens 1 zijn.")
if spring_design_mode not in ("fraction_of_baseline_hold", "manual"):
    raise ValueError("spring_design_mode moet 'fraction_of_baseline_hold' of 'manual' zijn.")
if not (0.0 < spring_max_assist_fraction < 1.0):
    raise ValueError("spring_max_assist_fraction moet tussen 0 en 1 liggen.")
if spring_direct_rate_per_spring_N_per_mm <= 0:
    raise ValueError("spring_direct_rate_per_spring_N_per_mm moet positief zijn.")
print(f"trekveer-case berekenen = {compute_spring_assist_case}")
print(f"directe trekveerstijfheid = {spring_direct_rate_per_spring_N_per_mm:.3f} N/mm per veer")


## Rigid-body kinematica

De punten en zwaartepunten worden op dezelfde manier gereconstrueerd als in Notebook 2. Punt K blijft een aparte puntmassa die aan link 8 gekoppeld is.


In [ ]:
def perp(v):
    v = np.asarray(v)
    return np.column_stack((-v[:, 1], v[:, 0])) if v.ndim == 2 else np.array([-v[1], v[0]])

def rigid_point(ref_pos, ref_vel, ref_acc, theta, omega, alpha_val, local_vector):
    local_vector = np.asarray(local_vector, dtype=float)
    r_global = np.column_stack((
        local_vector[0] * np.cos(theta) - local_vector[1] * np.sin(theta),
        local_vector[0] * np.sin(theta) + local_vector[1] * np.cos(theta),
    ))
    pos = ref_pos + r_global
    vel = ref_vel + omega[:, None] * perp(r_global)
    acc = ref_acc + alpha_val[:, None] * perp(r_global) - (omega[:, None] ** 2) * r_global
    return pos, vel, acc

zero = np.zeros(n_steps); zeros2 = np.zeros((n_steps, 2))
C_pos = zeros2.copy(); C_vel = zeros2.copy(); C_acc = zeros2.copy()
B_pos = np.column_stack((zero, -s)); B_vel = np.column_stack((zero, -ds)); B_acc = np.column_stack((zero, -dds))

D_pos, D_vel, D_acc = rigid_point(B_pos, B_vel, B_acc, theta3, dtheta3, ddtheta3, [r3a, 0.0])
E3_pos, _, _ = rigid_point(B_pos, B_vel, B_acc, theta3, dtheta3, ddtheta3, [r3a + r3b, 0.0])
E4_pos, _, _ = rigid_point(C_pos, C_vel, C_acc, theta4, dtheta4, ddtheta4, [r4a, 0.0])
E_pos = 0.5 * (E3_pos + E4_pos)
H_pos, H_vel, H_acc = rigid_point(C_pos, C_vel, C_acc, theta4, dtheta4, ddtheta4, [r4a + r4b, 0.0])
F_pos, F_vel, F_acc = rigid_point(D_pos, D_vel, D_acc, theta5, dtheta5, ddtheta5, [r5a, 0.0])
G5_pos, _, _ = rigid_point(D_pos, D_vel, D_acc, theta5, dtheta5, ddtheta5, [r5a + r5b, 0.0])
G7_pos, _, _ = rigid_point(H_pos, H_vel, H_acc, theta7, dtheta7, ddtheta7, [-r7a, 0.0])
G_pos = 0.5 * (G5_pos + G7_pos)
I_pos, I_vel, I_acc = rigid_point(F_pos, F_vel, F_acc, theta6, dtheta6, ddtheta6, [r6, 0.0])
J7_pos, _, _ = rigid_point(H_pos, H_vel, H_acc, theta7, dtheta7, ddtheta7, [r7b, 0.0])
J8_pos, _, _ = rigid_point(I_pos, I_vel, I_acc, theta8, dtheta8, ddtheta8, [r8a, 0.0])
J_pos = 0.5 * (J7_pos + J8_pos)
K_pos, K_vel, K_acc = rigid_point(I_pos, I_vel, I_acc, theta8, dtheta8, ddtheta8, [r8a + r8b, 0.0])

cg_pos, cg_vel, cg_acc, alpha = {}, {}, {}, {}
cg_pos[2] = B_pos.copy(); cg_vel[2] = B_vel.copy(); cg_acc[2] = B_acc.copy(); alpha[2] = zero.copy()
cg_pos[3], cg_vel[3], cg_acc[3] = rigid_point(B_pos, B_vel, B_acc, theta3, dtheta3, ddtheta3, [L3 / 2, 0.0]); alpha[3] = ddtheta3
cg_pos[4], cg_vel[4], cg_acc[4] = rigid_point(C_pos, C_vel, C_acc, theta4, dtheta4, ddtheta4, [L4 / 2, 0.0]); alpha[4] = ddtheta4
cg_pos[5], cg_vel[5], cg_acc[5] = rigid_point(D_pos, D_vel, D_acc, theta5, dtheta5, ddtheta5, [L5 / 2, 0.0]); alpha[5] = ddtheta5
cg_pos[6], cg_vel[6], cg_acc[6] = rigid_point(F_pos, F_vel, F_acc, theta6, dtheta6, ddtheta6, [L6 / 2, 0.0]); alpha[6] = ddtheta6
cg_pos[7], cg_vel[7], cg_acc[7] = rigid_point(H_pos, H_vel, H_acc, theta7, dtheta7, ddtheta7, [(r7b - r7a) / 2, 0.0]); alpha[7] = ddtheta7
cg_pos[8], cg_vel[8], cg_acc[8] = rigid_point(I_pos, I_vel, I_acc, theta8, dtheta8, ddtheta8, [L8 / 2, 0.0]); alpha[8] = ddtheta8

K_loaded = np.column_stack((Kx, Ky))
print("Rigid-body kinematica opgebouwd.")
print(f"max |K_reconstructie - K_Notebook1| = {np.max(np.linalg.norm(K_pos - K_loaded, axis=1)):.3e} m")
print(f"max sluitverschil E/G/J = {np.max(np.linalg.norm(E3_pos - E4_pos, axis=1)):.3e} / {np.max(np.linalg.norm(G5_pos - G7_pos, axis=1)):.3e} / {np.max(np.linalg.norm(J7_pos - J8_pos, axis=1)):.3e} m")


## Inverse dynamica

De onbekenden zijn dezelfde als in Notebook 3. De overdekkingsmassa zit als puntmassa in K. Een optionele trekveer wordt als bekende externe kracht op de schuiver toegevoegd, in globale positieve y-richting. Omdat `s` positief naar beneden is, helpt die kracht tijdens openen.


In [ ]:
def moment_2d(r, F):
    return r[0] * F[1] - r[1] * F[0]

def link_acc(link_id, k, static=False):
    return np.zeros(2) if static else cg_acc[link_id][k]

def link_alpha(link_id, k, static=False):
    return 0.0 if static else alpha[link_id][k]

def point_acc_K(k, static=False):
    return np.zeros(2) if static else K_acc[k]

def add_link_equations(A, b, row, m, J, a_cg, alpha_cg, cg, forces, known_forces=None, known_moments=None, point_loads=None, unknown_moments=None):
    known_forces = known_forces or []
    known_moments = known_moments or []
    point_loads = point_loads or []
    unknown_moments = unknown_moments or []
    row_fx, row_fy, row_m = row, row + 1, row + 2
    for fx_name, fy_name, sign, point in forces:
        r = point - cg
        if fx_name is not None:
            j = unknown_index[fx_name]
            A[row_fx, j] += sign
            A[row_m, j] += -sign * r[1]
        if fy_name is not None:
            j = unknown_index[fy_name]
            A[row_fy, j] += sign
            A[row_m, j] += sign * r[0]

    F_rhs = m * a_cg
    M_rhs = J * alpha_cg
    for moment_name, sign in unknown_moments:
        A[row_m, unknown_index[moment_name]] += sign

    for F_known, point in known_forces:
        r = point - cg
        F_rhs = F_rhs - F_known
        M_rhs = M_rhs - moment_2d(r, F_known)
    for M_known in known_moments:
        M_rhs = M_rhs - M_known
    for F_eff, point in point_loads:
        r = point - cg
        F_rhs = F_rhs + F_eff
        M_rhs = M_rhs + moment_2d(r, F_eff)

    b[row_fx] = F_rhs[0]
    b[row_fy] = F_rhs[1]
    b[row_m] = M_rhs
    return row + 3

def gravity_force(link_id):
    return masses[link_id] * g_vec

def build_inverse_dynamics_system(k, include_gravity=True, include_spring=False, spring_force_up=0.0, slider_friction_s=0.0, pin_moments_by_link=None, static=False):
    pin_moments_by_link = pin_moments_by_link or {}
    A = np.zeros((n_unknowns, n_unknowns)); b = np.zeros(n_unknowns); row = 0

    def known_forces_for(link_id):
        known = []
        if include_gravity:
            known.append((gravity_force(link_id), cg_pos[link_id][k]))
        if link_id == 2 and include_spring:
            known.append((np.array([0.0, spring_force_up]), B_pos[k]))
        if link_id == 2 and slider_friction_s != 0.0:
            known.append((np.array([0.0, -slider_friction_s]), B_pos[k]))
        return known

    def known_moments_for(link_id):
        M = pin_moments_by_link.get(link_id, 0.0)
        return [M] if M != 0.0 else []

    row = add_link_equations(A, b, row, masses[2], inertias[2], link_acc(2, k, static), link_alpha(2, k, static), cg_pos[2][k],
                             [("R_Ax", None, 1.0, B_pos[k]), (None, "F_act_y", 1.0, B_pos[k]), ("B_x", "B_y", 1.0, B_pos[k])],
                             known_forces=known_forces_for(2), known_moments=known_moments_for(2),
                             unknown_moments=[("M_A", 1.0)])
    row = add_link_equations(A, b, row, masses[3], inertias[3], link_acc(3, k, static), link_alpha(3, k, static), cg_pos[3][k],
                             [("B_x", "B_y", -1.0, B_pos[k]), ("D_x", "D_y", 1.0, D_pos[k]), ("E_x", "E_y", 1.0, E_pos[k])],
                             known_forces=known_forces_for(3), known_moments=known_moments_for(3))
    row = add_link_equations(A, b, row, masses[4], inertias[4], link_acc(4, k, static), link_alpha(4, k, static), cg_pos[4][k],
                             [("C_x", "C_y", 1.0, C_pos[k]), ("E_x", "E_y", -1.0, E_pos[k]), ("H_x", "H_y", 1.0, H_pos[k])],
                             known_forces=known_forces_for(4), known_moments=known_moments_for(4))
    row = add_link_equations(A, b, row, masses[5], inertias[5], link_acc(5, k, static), link_alpha(5, k, static), cg_pos[5][k],
                             [("D_x", "D_y", -1.0, D_pos[k]), ("F_x", "F_y", 1.0, F_pos[k]), ("G_x", "G_y", 1.0, G_pos[k])],
                             known_forces=known_forces_for(5), known_moments=known_moments_for(5))
    row = add_link_equations(A, b, row, masses[6], inertias[6], link_acc(6, k, static), link_alpha(6, k, static), cg_pos[6][k],
                             [("F_x", "F_y", -1.0, F_pos[k]), ("I_x", "I_y", 1.0, I_pos[k])],
                             known_forces=known_forces_for(6), known_moments=known_moments_for(6))
    row = add_link_equations(A, b, row, masses[7], inertias[7], link_acc(7, k, static), link_alpha(7, k, static), cg_pos[7][k],
                             [("G_x", "G_y", -1.0, G_pos[k]), ("H_x", "H_y", -1.0, H_pos[k]), ("J_x", "J_y", 1.0, J_pos[k])],
                             known_forces=known_forces_for(7), known_moments=known_moments_for(7))
    K_eff = payload_mass_K * (point_acc_K(k, static) - (g_vec if include_gravity else np.zeros(2)))
    row = add_link_equations(A, b, row, masses[8], inertias[8], link_acc(8, k, static), link_alpha(8, k, static), cg_pos[8][k],
                             [("I_x", "I_y", -1.0, I_pos[k]), ("J_x", "J_y", -1.0, J_pos[k])],
                             known_forces=known_forces_for(8), known_moments=known_moments_for(8),
                             point_loads=[(K_eff, K_pos[k])])
    if row != n_unknowns:
        raise RuntimeError(f"Verkeerd aantal vergelijkingen: {row} i.p.v. {n_unknowns}")
    return A, b


## Wrijvingsmodel

De schuiverwrijving werkt tegen de schuiverbeweging. De scharnierwrijving werkt tegen de relatieve hoeksnelheid tussen de twee verbonden links. Voor elk scharnier worden gelijke en tegengestelde momenten toegevoegd aan de twee verbonden lichamen.

Het model gebruikt Coulombwrijving van de vorm `mu*N` met een `tanh`-regularisatie rond nul snelheid. De actuator-effici?ntie wordt niet als extra mechanische kracht in het mechanisme gezet, maar pas achteraf gebruikt bij het motorvermogen.


In [ ]:
omega_link = {0: zero, 2: zero, 3: dtheta3, 4: dtheta4, 5: dtheta5, 6: dtheta6, 7: dtheta7, 8: dtheta8}
pin_joint_defs = [
    ("B", 3, 2, "B_x", "B_y"),
    ("C", 4, 0, "C_x", "C_y"),
    ("D", 5, 3, "D_x", "D_y"),
    ("E", 3, 4, "E_x", "E_y"),
    ("F", 6, 5, "F_x", "F_y"),
    ("G", 5, 7, "G_x", "G_y"),
    ("H", 4, 7, "H_x", "H_y"),
    ("I", 8, 6, "I_x", "I_y"),
    ("J", 7, 8, "J_x", "J_y"),
]
pin_joint_names = [item[0] for item in pin_joint_defs]


def reaction_norm_from_solution(w, fx_name, fy_name):
    return float(np.hypot(w[unknown_index[fx_name]], w[unknown_index[fy_name]]))


def compute_slider_friction_s(k, w_guess, velocity_sign=1.0):
    ds_eff = velocity_sign * ds[k]
    N_slider = abs(float(w_guess[unknown_index["R_Ax"]]))
    F_coulomb = -mu_slider * N_slider * np.tanh(ds_eff / v_eps)
    F_viscous = -c_slider * ds_eff
    return F_coulomb + F_viscous, N_slider


def compute_pin_friction(k, w_guess, velocity_sign=1.0):
    moments_by_link = {link_id: 0.0 for link_id in range(2, 9)}
    joint_moments = np.zeros(len(pin_joint_defs))
    joint_normal = np.zeros(len(pin_joint_defs))
    joint_power_loss = np.zeros(len(pin_joint_defs))
    if not include_pin_friction:
        return moments_by_link, joint_moments, joint_normal, joint_power_loss
    for j, (name, link_a, link_b, fx_name, fy_name) in enumerate(pin_joint_defs):
        N_joint = reaction_norm_from_solution(w_guess, fx_name, fy_name)
        omega_a = velocity_sign * omega_link[link_a][k]
        omega_b = velocity_sign * omega_link[link_b][k]
        omega_rel = omega_a - omega_b
        M_base = mu_pin * pin_radius * N_joint * np.tanh(omega_rel / omega_eps)
        M_on_a = -M_base
        M_on_b = M_base
        if link_a in moments_by_link:
            moments_by_link[link_a] += M_on_a
        if link_b in moments_by_link:
            moments_by_link[link_b] += M_on_b
        joint_moments[j] = M_on_a
        joint_normal[j] = N_joint
        joint_power_loss[j] = max(0.0, -(M_on_a * omega_a + M_on_b * omega_b))
    return moments_by_link, joint_moments, joint_normal, joint_power_loss


def solve_case(include_gravity=True, include_friction=False, include_spring=False, spring_force_up_time=None, static=False, velocity_sign=1.0):
    w_case = np.zeros((n_steps, n_unknowns))
    residual = np.zeros(n_steps); cond_dyn = np.zeros(n_steps)
    F_slider_friction_s = np.zeros(n_steps); N_slider = np.zeros(n_steps)
    joint_friction_moments = np.zeros((n_steps, len(pin_joint_defs)))
    joint_normal_forces = np.zeros((n_steps, len(pin_joint_defs)))
    joint_power_loss = np.zeros((n_steps, len(pin_joint_defs)))
    friction_iteration_delta = np.zeros(n_steps)

    for k in range(n_steps):
        spring_force_up_k = float(spring_force_up_time[k]) if (include_spring and spring_force_up_time is not None) else 0.0
        A0, b0 = build_inverse_dynamics_system(k, include_gravity=include_gravity, include_spring=include_spring, spring_force_up=spring_force_up_k, static=static)
        w_current = np.linalg.solve(A0, b0)
        if include_friction:
            last_delta = 0.0
            for _ in range(friction_iterations):
                F_slider, N_val = compute_slider_friction_s(k, w_current, velocity_sign=velocity_sign)
                M_by_link, M_joints, N_joints, P_loss = compute_pin_friction(k, w_current, velocity_sign=velocity_sign)
                A_k, b_k = build_inverse_dynamics_system(
                    k,
                    include_gravity=include_gravity,
                    include_spring=include_spring,
                    spring_force_up=spring_force_up_k,
                    slider_friction_s=F_slider,
                    pin_moments_by_link=M_by_link,
                    static=static,
                )
                w_next = np.linalg.solve(A_k, b_k)
                last_delta = np.linalg.norm(w_next - w_current)
                w_current = w_next
                if last_delta < friction_tol:
                    break
        else:
            F_slider = 0.0
            N_val = abs(float(w_current[unknown_index["R_Ax"]]))
            M_joints = np.zeros(len(pin_joint_defs))
            N_joints = np.array([reaction_norm_from_solution(w_current, item[3], item[4]) for item in pin_joint_defs])
            P_loss = np.zeros(len(pin_joint_defs))
            A_k, b_k = A0, b0
            last_delta = 0.0
        w_case[k] = w_current
        residual[k] = np.linalg.norm(A_k @ w_current - b_k)
        cond_dyn[k] = np.linalg.cond(A_k)
        F_slider_friction_s[k] = F_slider
        N_slider[k] = N_val
        joint_friction_moments[k] = M_joints
        joint_normal_forces[k] = N_joints
        joint_power_loss[k] = P_loss
        friction_iteration_delta[k] = last_delta
    return {
        "w": w_case,
        "residual": residual,
        "cond": cond_dyn,
        "F_slider_friction_s": F_slider_friction_s,
        "N_slider": N_slider,
        "joint_friction_moments": joint_friction_moments,
        "joint_normal_forces": joint_normal_forces,
        "joint_power_loss": joint_power_loss,
        "friction_iteration_delta": friction_iteration_delta,
        "velocity_sign": velocity_sign,
    }


## Baseline zonder trekveer

Eerst wordt de brede overdekking zonder veer opgelost. Deze case is de hoofdcase en vormt de referentie voor de motorbelasting en voor de eventuele trekveerassistentie.


In [ ]:
case_inertia_check = solve_case(include_gravity=False, include_friction=False)
case_gravity = solve_case(include_gravity=True, include_friction=False)
case_total = solve_case(include_gravity=True, include_friction=True, velocity_sign=1.0)
case_total_close = solve_case(include_gravity=True, include_friction=True, velocity_sign=-1.0)


def unpack_case(case):
    return {name: case["w"][:, idx] for name, idx in unknown_index.items()}


def cumulative_integral(time_values, y_values):
    out = np.zeros_like(y_values, dtype=float)
    out[1:] = np.cumsum(0.5 * (y_values[1:] + y_values[:-1]) * np.diff(time_values))
    return out


def close_time_series(position_order_values):
    return position_order_values[::-1]


def direction_energy_arrays(F_position_order, ds_position_order, reverse_for_time=False):
    P_position_order = F_position_order * ds_position_order
    if reverse_for_time:
        P_time = P_position_order[::-1]
        t_time = t - t[0]
    else:
        P_time = P_position_order
        t_time = t
    E_time = cumulative_integral(t_time, P_time)
    E_positive = float(np.trapezoid(np.maximum(P_time, 0.0), t_time))
    E_negative = float(np.trapezoid(-np.minimum(P_time, 0.0), t_time))
    E_net = float(np.trapezoid(P_time, t_time))
    return P_position_order, P_time, E_time, E_positive, E_negative, E_net


vars_inertia_check = unpack_case(case_inertia_check)
vars_gravity = unpack_case(case_gravity)
vars_total = unpack_case(case_total)
vars_total_close = unpack_case(case_total_close)

F_drive_s_inertia_check = -vars_inertia_check["F_act_y"]
F_drive_s_inertia = F_drive_s_inertia_check
F_drive_s_gravity = -vars_gravity["F_act_y"]
F_drive_s_total = -vars_total["F_act_y"]
F_drive_s_total_close_position = -vars_total_close["F_act_y"]
F_gravity_component = F_drive_s_gravity - F_drive_s_inertia
F_friction_component = F_drive_s_total - F_drive_s_gravity
F_friction_component_close_position = F_drive_s_total_close_position - F_drive_s_gravity

# De schuiverwrijvingskracht zelf werkt op de schuiver in positieve s-richting bij openen.
# In de benodigde aandrijfkracht verschijnt ze met tegengesteld teken.
F_slider_drive_component = -case_total["F_slider_friction_s"]
F_slider_drive_component_close_position = -case_total_close["F_slider_friction_s"]
F_pin_friction_component = F_friction_component - F_slider_drive_component
F_pin_friction_component_close_position = F_friction_component_close_position - F_slider_drive_component_close_position

active_motion_mask = np.abs(ds) > 1e-6
if not np.any(active_motion_mask):
    active_motion_mask = np.ones_like(ds, dtype=bool)

# De kinematica uit Notebook 1 kan openen of sluiten voorstellen. Deze loadcase
# maakt de fysieke labels expliciet: openen is altijd s_closed -> s_open,
# sluiten is altijd s_open -> s_closed. De tweede richting gebruikt dezelfde
# geometrische baan in omgekeerde tijdsrichting, met wrijving opnieuw opgelost
# voor de tegengestelde snelheid.
input_mean_ds = float(np.mean(ds[active_motion_mask]))
input_is_opening = input_mean_ds < 0.0
input_motion_label = "openen" if input_is_opening else "sluiten"
opposite_motion_label = "sluiten" if input_is_opening else "openen"
opposite_ds_position = -ds

F_drive_s_total_opposite_position = F_drive_s_total_close_position.copy()
F_friction_component_opposite_position = F_friction_component_close_position.copy()
F_slider_drive_component_opposite_position = F_slider_drive_component_close_position.copy()
F_pin_friction_component_opposite_position = F_pin_friction_component_close_position.copy()

s_motion = s[active_motion_mask]
sort_motion = np.argsort(s_motion)

P_current_position, P_current_time, E_current_time, E_positive_current, E_negative_current, E_net_current = direction_energy_arrays(
    F_drive_s_total, ds, reverse_for_time=False
)
P_opposite_position, P_opposite_time, E_opposite_time, E_positive_opposite, E_negative_opposite, E_net_opposite = direction_energy_arrays(
    F_drive_s_total_opposite_position, opposite_ds_position, reverse_for_time=True
)

if input_is_opening:
    t_open_profile = t.copy()
    s_open_profile = s.copy()
    ds_open_profile = ds.copy()
    dds_open_profile = dds.copy()
    F_drive_s_total_open = F_drive_s_total.copy()
    F_drive_s_total_open_position = F_drive_s_total.copy()
    F_drive_s_inertia_open = F_drive_s_inertia.copy()
    P_act_total_open_position = P_current_position
    P_act_total_open = P_current_time
    E_act_total_open = E_current_time
    E_positive_open = E_positive_current
    E_negative_open = E_negative_current
    E_net_open = E_net_current

    t_close_profile = t - t[0]
    s_close_profile = s[::-1]
    ds_close_profile = opposite_ds_position[::-1]
    dds_close_profile = dds[::-1]
    F_drive_s_total_close = F_drive_s_total_opposite_position[::-1]
    F_drive_s_total_close_position = F_drive_s_total_opposite_position.copy()
    F_drive_s_inertia_close = F_drive_s_inertia[::-1]
    P_act_total_close_position = P_opposite_position
    P_act_total_close = P_opposite_time
    E_act_total_close = E_opposite_time
    E_positive_close = E_positive_opposite
    E_negative_close = E_negative_opposite
    E_net_close = E_net_opposite
else:
    t_close_profile = t.copy()
    s_close_profile = s.copy()
    ds_close_profile = ds.copy()
    dds_close_profile = dds.copy()
    F_drive_s_total_close = F_drive_s_total.copy()
    F_drive_s_total_close_position = F_drive_s_total.copy()
    F_drive_s_inertia_close = F_drive_s_inertia.copy()
    P_act_total_close_position = P_current_position
    P_act_total_close = P_current_time
    E_act_total_close = E_current_time
    E_positive_close = E_positive_current
    E_negative_close = E_negative_current
    E_net_close = E_net_current

    t_open_profile = t - t[0]
    s_open_profile = s[::-1]
    ds_open_profile = opposite_ds_position[::-1]
    dds_open_profile = dds[::-1]
    F_drive_s_total_open = F_drive_s_total_opposite_position[::-1]
    F_drive_s_total_open_position = F_drive_s_total_opposite_position.copy()
    F_drive_s_inertia_open = F_drive_s_inertia[::-1]
    P_act_total_open_position = P_opposite_position
    P_act_total_open = P_opposite_time
    E_act_total_open = E_opposite_time
    E_positive_open = E_positive_opposite
    E_negative_open = E_negative_opposite
    E_net_open = E_net_opposite

F_friction_component_close_position = F_friction_component_opposite_position if input_is_opening else F_friction_component
F_slider_drive_component_close_position = F_slider_drive_component_opposite_position if input_is_opening else F_slider_drive_component
F_pin_friction_component_close_position = F_pin_friction_component_opposite_position if input_is_opening else F_pin_friction_component
closing_ds_position = ds_close_profile[::-1] if input_is_opening else ds_close_profile
closing_mask = np.abs(ds_close_profile) > 1e-6

motion_mask_open = np.abs(ds_open_profile) > 1e-6
motion_mask_close = np.abs(ds_close_profile) > 1e-6
force_peak_open = float(np.max(np.abs(F_drive_s_total_open[motion_mask_open])))
force_peak_close = float(np.max(np.abs(F_drive_s_total_close[motion_mask_close])))
power_peak_open = float(np.max(np.maximum(P_act_total_open, 0.0)))
power_peak_close = float(np.max(np.maximum(P_act_total_close, 0.0)))
motor_design_direction = "openen" if max(force_peak_open, power_peak_open) >= max(force_peak_close, power_peak_close) else "sluiten"

direction_names = np.array(["openen", "sluiten"])
direction_force_peaks = np.array([force_peak_open, force_peak_close])
direction_power_peaks = np.array([power_peak_open, power_peak_close])
direction_positive_energy = np.array([E_positive_open, E_positive_close])
direction_net_energy = np.array([E_net_open, E_net_close])

inertia_check_residual = float(np.max(case_inertia_check["residual"]))
max_diff_notebook2 = np.nan  # eigen overdekkingsmassa: geen rechtstreekse Notebook-2-vergelijking
friction_sign_check_open = np.max(case_total["F_slider_friction_s"][active_motion_mask] * ds[active_motion_mask])
friction_sign_check_close = np.max(case_total_close["F_slider_friction_s"][active_motion_mask] * closing_ds_position[active_motion_mask])

print("Dynamische controles:")
print(f"inertiecheck residu                 = {inertia_check_residual:.3e}")
print(f"max residu zwaartekracht             = {np.max(case_gravity['residual']):.3e}")
print(f"max residu totaal openen             = {np.max(case_total['residual']):.3e}")
print(f"max residu totaal sluiten            = {np.max(case_total_close['residual']):.3e}")
print(f"cond(A) totaal min/gem/max           = {np.min(case_total['cond']):.3e} / {np.mean(case_total['cond']):.3e} / {np.max(case_total['cond']):.3e}")
print(f"max iteratieverandering wrijving     = {np.max(case_total['friction_iteration_delta']):.3e}")
print(f"max(F_fric_slider_s * ds), openen    = {friction_sign_check_open:.3e} W-equivalent")
print(f"max(F_fric_slider_s * ds), sluiten   = {friction_sign_check_close:.3e} W-equivalent")
print()
print("Aandrijfkrachtcomponenten per mechanisme:")
print(f"max |F_s| inertie                    = {np.max(np.abs(F_drive_s_inertia)):.3f} N")
print(f"max |F_s| zwaartekrachtcomponent     = {np.max(np.abs(F_gravity_component)):.3f} N")
print(f"max |F_s| schuiverwrijving, openen   = {np.max(np.abs(F_slider_drive_component)):.3f} N")
print(f"max |F_s| schuiverwrijving, sluiten  = {np.max(np.abs(F_slider_drive_component_close_position)):.3f} N")
print(f"max |F_s| totaal openen              = {force_peak_open:.3f} N")
print(f"max |F_s| totaal sluiten             = {force_peak_close:.3f} N")
print(f"max positief vermogen openen         = {power_peak_open:.3f} W")
print(f"max positief vermogen sluiten        = {power_peak_close:.3f} W")
print(f"ontwerprichtingskandidaat            = {motor_design_direction}")
print(f"max |F_s| totaal, alle mechanismen   = {mechanism_count_total*max(force_peak_open, force_peak_close):.3f} N")


## Validatie via energiebalans

De inverse-dynamica oplossing wordt gevalideerd met behulp van de **arbeid-energiestelling**:

$$P_\text{actuator}(t) = F_s(t)\,\dot{s}(t) = \frac{d}{dt}\!\sum_i\!\left(\tfrac{1}{2}m_i v_{cg,i}^2 + \tfrac{1}{2}J_i\omega_i^2\right) + \sum_i m_i g\, v_{cg,i,y} + P_\text{wrijving}(t)$$

Het linkerlid is het mechanische ingangsvermogen (actuator). Het rechterlid is de som van:
- de tijdsafgeleide van de totale kinetische energie (translatie + rotatie van alle bewegende links en puntmassa K),
- het vermogen geleverd tegen de zwaartekracht ($m_i g v_{cg,i,y}$),
- het gedissipeerde vermogen door wrijving ($P_\text{wrijving}$, positief = verlies).

Als de implementatie correct is, moeten linker- en rechterlid op elk tijdstip overeenkomen tot machineprecisie (na afronding).


In [ ]:
# Energiebalans-validatie voor de totale oplossing (zwaartekracht + wrijving)
# LHS: actuatorvermogen = F_s * ds
P_actuator_val = F_drive_s_total * ds

# RHS: dEkin/dt + P_zwaartekracht + P_wrijving
# Kinetische energie per link: Ekin = 0.5*m*||v_cg||^2 + 0.5*J*omega^2
# We berekenen dEkin/dt numeriek via centrale differenties (behalve de grenzen).
links_val = [
    (masses[2], inertias[2], cg_vel[2], zero),
    (masses[3], inertias[3], cg_vel[3], dtheta3),
    (masses[4], inertias[4], cg_vel[4], dtheta4),
    (masses[5], inertias[5], cg_vel[5], dtheta5),
    (masses[6], inertias[6], cg_vel[6], dtheta6),
    (masses[7], inertias[7], cg_vel[7], dtheta7),
    (masses[8], inertias[8], cg_vel[8], dtheta8),
]

Ekin = np.zeros(n_steps)
P_gravity_val = np.zeros(n_steps)
for m_i, J_i, v_cg_i, omega_i in links_val:
    Ekin += 0.5 * m_i * np.sum(v_cg_i**2, axis=1) + 0.5 * J_i * omega_i**2
    P_gravity_val += m_i * g * v_cg_i[:, 1]

# Puntmassa K (gekoppeld aan link 8, geen rotatie-inertie)
Ekin += 0.5 * payload_mass_K * np.sum(K_vel**2, axis=1)
P_gravity_val += payload_mass_K * g * K_vel[:, 1]

# Numerieke afgeleide van de kinetische energie (centrale differenties)
dEkin_dt = np.gradient(Ekin, t)

# Gedissipeerd wrijvingsvermogen (altijd >= 0: schuiver + scharnieren)
P_friction_val = -case_total["F_slider_friction_s"] * ds  # schuiverwrijving: F_fric * (-ds) >= 0
P_friction_val += np.sum(case_total["joint_power_loss"], axis=1)

# Energiebalans: LHS - RHS
energy_balance_error = P_actuator_val - (dEkin_dt + P_gravity_val + P_friction_val)

# Relatieve fout (genormeerd op |LHS| waar die groot genoeg is)
denom_val = np.where(np.abs(P_actuator_val) > 1e-3, P_actuator_val, np.nan)
energy_balance_rel_error = np.abs(energy_balance_error / denom_val)

fig_eb, ax_eb = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
fig_eb.suptitle("Energiebalans-validatie (arbeid-energiestelling)")

ax_eb[0].plot(t, np.abs(energy_balance_error))
ax_eb[0].set_title("Absolute fout $|P_{act} - (\\dot{E}_{kin} + P_{zwk} + P_{wrijv})|$")
ax_eb[0].set_xlabel("t [s]"); ax_eb[0].set_ylabel("fout [W]")
ax_eb[0].set_yscale("log"); ax_eb[0].grid(True)

ax_eb[1].plot(t, energy_balance_rel_error)
ax_eb[1].set_title("Relatieve fout (alleen tijdens beweging)")
ax_eb[1].set_xlabel("t [s]"); ax_eb[1].set_ylabel("relatieve fout [-]")
ax_eb[1].set_yscale("log"); ax_eb[1].grid(True)

plt.show()

# Samenvatting
valid_mask = np.isfinite(energy_balance_rel_error)
print("Energiebalans-validatie:")
print(f"max absolute fout  = {np.max(np.abs(energy_balance_error)):.3e} W")
if np.any(valid_mask):
    print(f"max relatieve fout = {np.nanmax(energy_balance_rel_error):.3e}")
print("(Kleine fouten komen door numerieke differentiatie van Ekin; machineprecisie bij analytische afleiding.)")


## Krachtdecompositie

De volgende grafieken tonen de grootheden die nodig zijn voor de ontwerpkeuze: aandrijfkracht, wrijvingsbijdrage, actuatorvermogen en de evolutie tegenover de schuiverpositie.


In [ ]:
fig_force, ax_force = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
fig_force.suptitle("Aandrijfkracht met zwaartekracht en wrijving")

ax_force[0, 0].plot(t, F_drive_s_total, label="totaal", color="tab:blue")
ax_force[0, 0].plot(t, F_drive_s_gravity, label="zonder wrijving", color="tab:orange", alpha=0.8)
ax_force[0, 0].plot(t, F_drive_s_inertia, label="alleen inertie", color="tab:green", alpha=0.8)
ax_force[0, 0].set_title("Benodigde schuifkracht in tijd")
ax_force[0, 0].set_xlabel("t [s]"); ax_force[0, 0].set_ylabel("F_s [N]")
ax_force[0, 0].grid(True); ax_force[0, 0].legend()

ax_force[0, 1].plot(t, F_drive_s_inertia, label="inertie")
ax_force[0, 1].plot(t, F_gravity_component, label="zwaartekracht")
ax_force[0, 1].plot(t, F_slider_drive_component, label="schuiverwrijving")
ax_force[0, 1].plot(t, F_pin_friction_component, label="scharnierwrijving/effect")
ax_force[0, 1].set_title("Componenten in tijd")
ax_force[0, 1].set_xlabel("t [s]"); ax_force[0, 1].set_ylabel("F_s [N]")
ax_force[0, 1].grid(True); ax_force[0, 1].legend()

ax_force[1, 0].plot(s_motion[sort_motion], F_drive_s_inertia[active_motion_mask][sort_motion], label="inertie")
ax_force[1, 0].plot(s_motion[sort_motion], F_gravity_component[active_motion_mask][sort_motion], label="zwaartekracht")
ax_force[1, 0].plot(s_motion[sort_motion], F_slider_drive_component[active_motion_mask][sort_motion], label="schuiverwrijving")
ax_force[1, 0].plot(s_motion[sort_motion], F_pin_friction_component[active_motion_mask][sort_motion], label="scharnierwrijving/effect")
ax_force[1, 0].set_title("Componenten tegenover s tijdens beweging")
ax_force[1, 0].set_xlabel("s [m]"); ax_force[1, 0].set_ylabel("F_s [N]")
ax_force[1, 0].grid(True); ax_force[1, 0].legend()

ax_force[1, 1].plot(t, case_total["N_slider"], label="normaalkracht schuiver")
ax_force[1, 1].plot(t, -F_slider_drive_component, label="schuiverwrijving op schuiver")
ax_force[1, 1].set_title("Schuivergeleiding")
ax_force[1, 1].set_xlabel("t [s]"); ax_force[1, 1].set_ylabel("N")
ax_force[1, 1].grid(True); ax_force[1, 1].legend()
plt.show()

fig_pin, ax_pin = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
fig_pin.suptitle("Scharnierwrijving")
for j, name in enumerate(pin_joint_names):
    ax_pin[0].plot(t, case_total["joint_friction_moments"][:, j], label=name)
ax_pin[0].set_title("Wrijvingsmoment per scharnier")
ax_pin[0].set_xlabel("t [s]"); ax_pin[0].set_ylabel("M [Nm]")
ax_pin[0].grid(True); ax_pin[0].legend(ncol=3, fontsize=8)

ax_pin[1].plot(t, np.sum(case_total["joint_power_loss"], axis=1), label="som pinverliezen")
ax_pin[1].set_title("Dissipatie in scharnieren")
ax_pin[1].set_xlabel("t [s]"); ax_pin[1].set_ylabel("P_loss [W]")
ax_pin[1].grid(True); ax_pin[1].legend()
plt.show()


## Statische houdanalyse

Deze analyse is nodig omdat de paraplu in open stand of in tussenstanden moet kunnen blijven staan. In een statische stand zijn alle snelheden en versnellingen nul, maar zwaartekracht blijft werken. De belangrijke grootheid is daarom de vereiste houdkracht `F_hold`.

De curve met statische schuivercapaciteit is alleen een indicatie van hoeveel wrijving de geleiding theoretisch zou kunnen leveren: `mu_static * |R_Ax|`. Dat is geen goed ontwerpcriterium, want wrijving varieert door slijtage, vuil en smering. Voor het ontwerp moet een rem of mechanische vergrendeling (tandriem is niet-zelfremmend) de houdkracht veilig kunnen opnemen.


In [ ]:
hold_curve_indices = np.argsort(s)
hold_s_curve = s[hold_curve_indices]
F_hold_s_curve = np.zeros_like(hold_s_curve)
R_Ax_hold_curve = np.zeros_like(hold_s_curve)
C_x_hold_curve = np.zeros_like(hold_s_curve)
C_y_hold_curve = np.zeros_like(hold_s_curve)

for out_i, k in enumerate(hold_curve_indices):
    A_hold, b_hold = build_inverse_dynamics_system(k, include_gravity=True, static=True)
    w_hold = np.linalg.solve(A_hold, b_hold)
    F_hold_s_curve[out_i] = -w_hold[unknown_index["F_act_y"]]
    R_Ax_hold_curve[out_i] = w_hold[unknown_index["R_Ax"]]
    C_x_hold_curve[out_i] = w_hold[unknown_index["C_x"]]
    C_y_hold_curve[out_i] = w_hold[unknown_index["C_y"]]

static_slider_capacity_curve = mu_slider_static * np.abs(R_Ax_hold_curve)
T_hold_motor_curve = F_hold_s_curve * drive_travel_per_rev_nb3 / (2 * np.pi * actuator_efficiency)
T_hold_lock_required_curve = actuator_safety_factor * np.abs(F_hold_s_curve) * drive_travel_per_rev_nb3 / (2 * np.pi * actuator_efficiency)

sample_targets = np.linspace(np.min(s), np.max(s), 5)
sample_indices_curve = np.array([int(np.argmin(np.abs(hold_s_curve - target))) for target in sample_targets])
hold_s_values = hold_s_curve[sample_indices_curve]
F_hold_s = F_hold_s_curve[sample_indices_curve]
R_Ax_hold = R_Ax_hold_curve[sample_indices_curve]
static_slider_capacity = static_slider_capacity_curve[sample_indices_curve]
T_hold_motor = T_hold_motor_curve[sample_indices_curve]
T_hold_lock_required = T_hold_lock_required_curve[sample_indices_curve]

open_idx_curve = int(np.argmin(hold_s_curve))
closed_idx_curve = int(np.argmax(hold_s_curve))
open_idx = int(np.argmin(hold_s_values))
closed_idx = int(np.argmax(hold_s_values))

print("Statische houdanalyse:")
for i, s_val in enumerate(hold_s_values):
    label = "open" if i == open_idx else ("gesloten" if i == closed_idx else "tussenstand")
    print(f"s = {s_val:.3f} m ({label:10s}) | F_hold = {F_hold_s[i]: .2f} N | theoretische schuiverwrijvingsgrens = {static_slider_capacity[i]:.2f} N")
print()
print(f"open stand: |F_hold| = {abs(F_hold_s_curve[open_idx_curve]):.2f} N")
print(f"open stand: rem-/poeliekoppel met safety, zonder rekenen op wrijving = {T_hold_lock_required_curve[open_idx_curve]:.4f} Nm")

fig_hold, ax_hold = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
fig_hold.suptitle("Statische houdkracht")
ax_hold[0].plot(hold_s_curve, F_hold_s_curve, label="vereiste houdkracht")
ax_hold[0].axhline(0.0, color="black", linewidth=0.8)
ax_hold[0].set_xlabel("s [m]"); ax_hold[0].set_ylabel("F_hold,s [N]")
ax_hold[0].set_title("Houdkracht door zwaartekracht")
ax_hold[0].grid(True); ax_hold[0].legend()

ax_hold[1].plot(hold_s_curve, np.abs(F_hold_s_curve), label="|F_hold|")
ax_hold[1].plot(hold_s_curve, static_slider_capacity_curve, "--", label="theoretische schuiverwrijvingsgrens")
ax_hold[1].set_xlabel("s [m]"); ax_hold[1].set_ylabel("kracht [N]")
ax_hold[1].set_title("Niet ontwerpen op toevallige wrijving")
ax_hold[1].grid(True); ax_hold[1].legend()
plt.show()


## Vermogen en actuatorbelasting

De actuator volgt het traject `s_ref(t)` uit Notebook 1. De kracht volgt uit de inverse dynamica en het mechanische vermogen aan de schuiver is `P = F_s ds`. Positief vermogen betekent dat de actuator energie aan het mechanisme levert. Negatief vermogen betekent dat het mechanisme de aandrijving zou terugduwen of dat er geremd moet worden.

Het getoonde vermogen in de grafiek is mechanisch vermogen aan de schuiver. In de geprinte motorwaarden wordt het actuatorrendement en de veiligheidsfactor gebruikt voor een eerste motordimensionering.


In [ ]:
F_required_s_total = F_drive_s_total.copy()
P_act_total = P_act_total_open.copy()
E_act_total = E_act_total_open.copy()

motor_speed_rps = ds / drive_travel_per_rev_nb3
motor_speed_rpm = 60.0 * motor_speed_rps
T_motor_total = F_required_s_total * drive_travel_per_rev_nb3 / (2 * np.pi * actuator_efficiency)

motion_mask = np.abs(ds) > 1e-8
if not np.any(motion_mask):
    motion_mask = np.ones_like(ds, dtype=bool)

P_positive = np.maximum(P_act_total, 0.0)
P_negative = np.minimum(P_act_total, 0.0)
P_positive_peak = actuator_safety_factor * np.max(P_positive) / actuator_efficiency
P_regen_peak = actuator_safety_factor * abs(np.min(P_negative)) * actuator_efficiency
T_motor_peak = actuator_safety_factor * np.max(np.abs(T_motor_total))
T_motor_rms = actuator_safety_factor * np.sqrt(np.mean(T_motor_total[motion_mask]**2))
F_total_peak = np.max(np.abs(F_required_s_total[motion_mask]))
F_total_rms = np.sqrt(np.mean(F_required_s_total[motion_mask]**2))
energy_net_total = E_act_total[-1] - E_act_total[0]
energy_fluctuation_total = np.max(E_act_total) - np.min(E_act_total)

print("Actuatoranalyse met zwaartekracht en wrijving:")
print(f"piek schuifkracht openen             = {force_peak_open:.2f} N")
print(f"piek schuifkracht sluiten            = {force_peak_close:.2f} N")
print(f"RMS schuifkracht openen              = {F_total_rms:.2f} N")
print(f"max positief actuatorvermogen openen = {power_peak_open:.2f} W")
print(f"max positief actuatorvermogen sluiten= {power_peak_close:.2f} W")
print(f"positieve energie openen             = {E_positive_open:.2f} J")
print(f"positieve energie sluiten            = {E_positive_close:.2f} J")
print(f"netto mechanische energie openen     = {E_net_open:.2f} J")
print(f"netto mechanische energie sluiten    = {E_net_close:.2f} J")
print(f"energiefluctuatie openen             = {energy_fluctuation_total:.2f} J")
print(f"piek motortoerental                  = {np.max(np.abs(motor_speed_rpm)):.1f} rpm")
print(f"piek motorkoppel incl. safety        = {T_motor_peak:.4f} Nm")
print(f"RMS motorkoppel incl. safety         = {T_motor_rms:.4f} Nm")

fig_act, ax_act = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
fig_act.suptitle("Actuatorbelasting openen en sluiten")
ax_act[0, 0].plot(t_open_profile, s_open_profile, label="openen")
ax_act[0, 0].plot(t_close_profile, s_close_profile, label="sluiten")
ax_act[0, 0].set_title("Schuiverpositie"); ax_act[0, 0].set_xlabel("t [s]"); ax_act[0, 0].set_ylabel("s [m]"); ax_act[0, 0].grid(True); ax_act[0, 0].legend()
ax_act[0, 1].plot(s_open_profile, F_drive_s_total_open, label="openen")
ax_act[0, 1].plot(s_close_profile, F_drive_s_total_close, label="sluiten")
ax_act[0, 1].invert_xaxis()
ax_act[0, 1].set_title("Benodigde schuifkracht"); ax_act[0, 1].set_xlabel("s [m]"); ax_act[0, 1].set_ylabel("F_s [N]"); ax_act[0, 1].grid(True); ax_act[0, 1].legend()
ax_act[1, 0].plot(t_open_profile, P_act_total_open, label="openen")
ax_act[1, 0].plot(t_close_profile, P_act_total_close, label="sluiten")
ax_act[1, 0].axhline(0.0, color="black", linewidth=0.8)
ax_act[1, 0].set_title("Actuatorvermogen"); ax_act[1, 0].set_xlabel("t [s]"); ax_act[1, 0].set_ylabel("P [W]"); ax_act[1, 0].grid(True); ax_act[1, 0].legend()
ax_act[1, 1].plot(t_open_profile, E_act_total_open, label="openen")
ax_act[1, 1].plot(t_close_profile, E_act_total_close, label="sluiten")
ax_act[1, 1].set_title("Cumulatieve actuatorarbeid"); ax_act[1, 1].set_xlabel("t [s]"); ax_act[1, 1].set_ylabel("E [J]"); ax_act[1, 1].grid(True); ax_act[1, 1].legend()
plt.show()


## Arbeids-surplus en motordimensionering

Het arbeids-surplus $A_\text{max}$ is een maat voor de energiebuffer die de motor of regelelektronica moet kunnen leveren of opnemen binnen een bewegingscyclus:

$$A_\text{max} = \max_t \int_0^t \!\bigl(P(t') - \bar P\bigr)\,dt' \;-\; \min_t \int_0^t \!\bigl(P(t') - \bar P\bigr)\,dt'$$

Hier is $\bar P$ het tijdsgemiddelde van het actuatorvermogen en $P(t) = F_s(t)\,\dot s(t)$ het ogenblikkelijke ingangsvermogen. De grootheid $A_\text{max}$ vertelt hoeveel mechanische energie tijdelijk gebufferd moet worden als de belasting boven of onder het gemiddelde ligt.

$A_\text{max}$ hangt af van de gekozen bewegingswet en van dissipatie door wrijving. Hier wordt de waarde berekend voor de volledige belasting: inertie, zwaartekracht en wrijving.


In [ ]:
# Arbeids-surplusop basis van de volledige belastingskurve
P_load_full = F_required_s_total * ds          # ogenblikkelijk ingangsvermogen [W]
P_avg_full  = P_load_full.mean()               # gemiddelde over de volledige simulatie
P_peak_full = np.abs(P_load_full).max()
P_rms_full  = np.sqrt(np.mean(P_load_full**2))
F_peak_full = np.abs(F_required_s_total[motion_mask]).max()
F_rms_full  = np.sqrt(np.mean(F_required_s_total[motion_mask]**2))

# Arbeids-surplus: cumulatieve integraal van (P - P_gem)
A_theta_full = np.cumsum((P_load_full - P_avg_full) * Ts)   # [J]
A_max_full   = A_theta_full.max() - A_theta_full.min()       # vereiste energiebuffer [J]

# Aandrijfkoppel via tandriem/poelie (definitieve motorselectie in NB4)
T_motor_peak_sizing = actuator_safety_factor * F_peak_full * drive_travel_per_rev_nb3 / (2 * np.pi * actuator_efficiency)
T_motor_rms_sizing  = actuator_safety_factor * F_rms_full  * drive_travel_per_rev_nb3 / (2 * np.pi * actuator_efficiency)
P_motor_rated_req   = 1.5 * P_rms_full
P_motor_peak_req    = 1.3 * P_peak_full

print("=" * 62)
print("  ARBEIDS-SURPLUS EN MOTORDIMENSIONERING ")
print("=" * 62)
print(f"Gemiddeld ingangsvermogen  P_avg  = {P_avg_full:.3f} W")
print(f"Piek    ingangsvermogen    P_peak = {P_peak_full:.3f} W")
print(f"RMS     ingangsvermogen    P_rms  = {P_rms_full:.3f} W")
print(f"Piek    aandrijfkracht     F_peak = {F_peak_full:.3f} N")
print(f"RMS     aandrijfkracht     F_rms  = {F_rms_full:.3f} N")
print()
print(f"Arbeids-surplus  A_max = {A_max_full:.4f} J   (energiebuffer vereist)")
print()
print(f"Tandriem/poelie quick check (r_ref = {drive_pulley_radius_nb3*1000:.1f} mm, eta = {actuator_efficiency:.2f}):")
print(f"  Piek motorkoppel  (x{actuator_safety_factor:.1f}) = {T_motor_peak_sizing:.4f} Nm")
print(f"  RMS  motorkoppel  (x{actuator_safety_factor:.1f}) = {T_motor_rms_sizing:.4f} Nm")
print()
print("Aanbevolen motorspecificaties:")
print(f"  Continu nominaal vermogen  >= {P_motor_rated_req:.1f} W   (= 1.5 x P_rms)")
print(f"  Piekvermogen-capaciteit    >= {P_motor_peak_req:.1f} W   (= 1.3 x P_peak)")
print(f"  Nominaal koppel            >= {T_motor_rms_sizing:.4f} Nm")
print(f"  Piekkoppel-capaciteit      >= {T_motor_peak_sizing:.4f} Nm")

# Plot arbeids-surplus (arbeids-surplusmethode)
fig_amax, ax_amax = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
fig_amax.suptitle("Arbeids-surplus — volledige belasting")

ax_amax[0].plot(t, P_load_full, label="P(t)")
ax_amax[0].axhline(P_avg_full, color="gray", ls="--", lw=1, label=f"P_gem = {P_avg_full:.2f} W")
ax_amax[0].set_title("Ingangsvermogen")
ax_amax[0].set_xlabel("t [s]"); ax_amax[0].set_ylabel("P [W]")
ax_amax[0].grid(True); ax_amax[0].legend()

ax_amax[1].plot(t, A_theta_full, "k", lw=1.5)
ax_amax[1].axhline(A_theta_full.max(), color="red",  ls="--", lw=1,
                   label=f"max = {A_theta_full.max():+.3f} J")
ax_amax[1].axhline(A_theta_full.min(), color="blue", ls="--", lw=1,
                   label=f"min = {A_theta_full.min():+.3f} J")
ax_amax[1].fill_between(t, A_theta_full, A_theta_full.min(), alpha=0.12, color="red")
ax_amax[1].set_title(f"Arbeids-surplus: $A_{{max}}$ = {A_max_full:.3f} J")
ax_amax[1].set_xlabel("t [s]"); ax_amax[1].set_ylabel("$\\int(P - \\bar P)\\,dt$ [J]")
ax_amax[1].grid(True); ax_amax[1].legend()

plt.show()


## Framebelasting en onbalans

Onbalans en framebelasting zijn niet hetzelfde. De inertiele referentie hoort bij versnellingen en is klein omdat het traject traag is. Met zwaartekracht krijgt het frame daarbovenop een bijna statische gewichtslast. De afzonderlijke steunreacties aan de schuiver en aan C kunnen groot zijn door de mechanische overbrenging, zelfs als de netto resultante op het volledige frame ongeveer gelijk is aan het totale gewicht.


In [ ]:
F_slider_global = np.column_stack((np.zeros(n_steps), -case_total["F_slider_friction_s"]))
F_A_total = np.column_stack((vars_total["R_Ax"], vars_total["F_act_y"])) + F_slider_global
F_C_total = np.column_stack((vars_total["C_x"], vars_total["C_y"]))
F_frame_total = F_A_total + F_C_total
F_frame_total_norm = np.linalg.norm(F_frame_total, axis=1)
F_C_total_norm = np.linalg.norm(F_C_total, axis=1)
F_A_total_norm = np.linalg.norm(F_A_total, axis=1)
total_weight = total_model_mass * g

F_shak_norm_inertia = np.abs(F_drive_s_inertia)

fig_frame, ax_frame = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
fig_frame.suptitle("Framebelasting met zwaartekracht en wrijving")
ax_frame[0, 0].plot(t, F_A_total_norm, label="|A| schuiver/actuator")
ax_frame[0, 0].plot(t, F_C_total_norm, label="|C| framepunt")
ax_frame[0, 0].set_title("Afzonderlijke steunreacties")
ax_frame[0, 0].set_xlabel("t [s]"); ax_frame[0, 0].set_ylabel("N")
ax_frame[0, 0].grid(True); ax_frame[0, 0].legend()

ax_frame[0, 1].plot(t, F_A_total[:, 0], label="A_x")
ax_frame[0, 1].plot(t, F_A_total[:, 1], label="A_y")
ax_frame[0, 1].plot(t, F_C_total[:, 0], label="C_x")
ax_frame[0, 1].plot(t, F_C_total[:, 1], label="C_y")
ax_frame[0, 1].set_title("Componenten van steunreacties")
ax_frame[0, 1].set_xlabel("t [s]"); ax_frame[0, 1].set_ylabel("N")
ax_frame[0, 1].grid(True); ax_frame[0, 1].legend(ncol=2)

ax_frame[1, 0].plot(t, F_shak_norm_inertia, label="inertiele referentie")
ax_frame[1, 0].plot(t, F_frame_total_norm, label="netto frame-resultante")
ax_frame[1, 0].axhline(total_weight, color="black", linestyle="--", linewidth=1.0, label="totaal gewicht")
ax_frame[1, 0].set_title("Onbalans versus netto frame-resultante")
ax_frame[1, 0].set_xlabel("t [s]"); ax_frame[1, 0].set_ylabel("N")
ax_frame[1, 0].grid(True); ax_frame[1, 0].legend()

ax_frame[1, 1].plot(s_motion[sort_motion], F_A_total_norm[active_motion_mask][sort_motion], label="|A|")
ax_frame[1, 1].plot(s_motion[sort_motion], F_C_total_norm[active_motion_mask][sort_motion], label="|C|")
ax_frame[1, 1].plot(s_motion[sort_motion], np.abs(F_required_s_total[active_motion_mask][sort_motion]), label="|F_s totaal|")
ax_frame[1, 1].set_title("Belastingen tegenover s tijdens beweging")
ax_frame[1, 1].set_xlabel("s [m]"); ax_frame[1, 1].set_ylabel("N")
ax_frame[1, 1].grid(True); ax_frame[1, 1].legend()
plt.show()

print("Framebelasting:")
print(f"totaal gewicht model       = {total_weight:.2f} N")
print(f"max netto frame-resultante = {np.max(F_frame_total_norm):.2f} N")
print(f"max frame-reactie aan C    = {np.max(F_C_total_norm):.2f} N")
print(f"max belasting aan schuiver = {np.max(F_A_total_norm):.2f} N")


## Mast- en schuiverbelasting

De aandrijving levert alleen de verticale kracht langs de schuiver. De grote horizontale reacties aan de schuiver en aan punt C zijn lokale constructiekrachten: die moeten door de schuivergeleiding, mast, muurbeugels of frame opgenomen worden. Daarom wordt hier apart een eerste-orde controle gemaakt van het krachtkoppel tussen schuiver en vast framepunt.


In [ ]:
# ============================================================
# Mast- en schuiverbelasting
# ============================================================

mast_bracket_spacing_candidates = np.array([0.40, 0.60, 0.80, 1.00, 1.50, 2.00])
mast_reference_bracket_spacing = 1.00

# Eerste dimensionering van de geleiding/collar en mast. Dit is geen detailontwerp
# van bouten, lassen of fundering, maar geeft wel de juiste orde van grootte.
guide_safety_factor = 2.0
loaded_roller_count = 4
guide_load_sharing_candidates = np.array([2, 4, 6, 8])
guide_reference_roller_capacity = 3000.0  # [N] orde-grootte voor een robuuste rol/glijblok
mast_material_name = "staal S235"
mast_yield_strength = 235e6
mast_stress_safety_factor = 1.5
mast_allowable_stress = mast_yield_strength / mast_stress_safety_factor
mast_density = 7850.0
mast_min_preferred_width = 0.10
mast_profile_catalog = {
    "80x80x5":   dict(width=0.080, height=0.080, thickness=0.005),
    "100x100x5": dict(width=0.100, height=0.100, thickness=0.005),
    "120x120x6": dict(width=0.120, height=0.120, thickness=0.006),
    "150x150x6": dict(width=0.150, height=0.150, thickness=0.006),
}


def evaluate_mast_guide_design(R_Ax_curve, F_drive_curve, frame_x_curve):
    R_Ax_curve = np.asarray(R_Ax_curve, dtype=float)
    F_drive_curve = np.asarray(F_drive_curve, dtype=float)
    frame_x_curve = np.asarray(frame_x_curve, dtype=float)

    mast_couple = np.abs(R_Ax_curve * s)
    idx_peak = int(np.argmax(mast_couple))
    M_peak = float(mast_couple[idx_peak])
    s_peak = float(s[idx_peak])
    side_peak = float(np.max(np.abs(R_Ax_curve)))
    frame_x_peak = float(np.max(np.abs(frame_x_curve)))
    drive_peak = float(np.max(np.abs(F_drive_curve[active_motion_mask])))
    ratio = side_peak / drive_peak if drive_peak > 1e-9 else np.inf

    bracket_force = M_peak / mast_bracket_spacing_candidates + 0.5 * frame_x_peak
    reference_bracket_force = float(M_peak / mast_reference_bracket_spacing + 0.5 * frame_x_peak)
    F_guide_design = guide_safety_factor * side_peak
    F_roller_design = F_guide_design / loaded_roller_count
    guide_force_per_support = F_guide_design / guide_load_sharing_candidates

    profile_names = []
    profile_mass_per_m = []
    profile_section_modulus = []
    profile_sigma = []
    profile_util = []
    profile_ok = []
    profile_widths = []
    for name, profile in mast_profile_catalog.items():
        width = profile["width"]
        height = profile["height"]
        thickness = profile["thickness"]
        area = rectangular_tube_area(width, height, thickness)
        I_strong = rectangular_tube_I_vertical(width, height, thickness)
        W_strong = I_strong / (0.5 * height)
        sigma = M_peak / W_strong
        util = sigma / mast_allowable_stress
        profile_names.append(name)
        profile_mass_per_m.append(area * mast_density)
        profile_section_modulus.append(W_strong)
        profile_sigma.append(sigma)
        profile_util.append(util)
        profile_ok.append(util <= 1.0)
        profile_widths.append(width)

    profile_names = np.array(profile_names, dtype=str)
    profile_mass_per_m = np.array(profile_mass_per_m, dtype=float)
    profile_section_modulus = np.array(profile_section_modulus, dtype=float)
    profile_sigma = np.array(profile_sigma, dtype=float)
    profile_util = np.array(profile_util, dtype=float)
    profile_ok = np.array(profile_ok, dtype=bool)
    profile_widths = np.array(profile_widths, dtype=float)
    preferred_mask = profile_ok & (profile_widths >= mast_min_preferred_width)
    if np.any(preferred_mask):
        selected_idx = int(np.where(preferred_mask)[0][0])
    elif np.any(profile_ok):
        selected_idx = int(np.where(profile_ok)[0][0])
    else:
        selected_idx = int(np.argmin(profile_util))

    return dict(
        mast_couple=mast_couple,
        idx_peak=idx_peak,
        M_peak=M_peak,
        s_peak=s_peak,
        side_peak=side_peak,
        frame_x_peak=frame_x_peak,
        drive_peak=drive_peak,
        ratio=ratio,
        bracket_force=bracket_force,
        reference_bracket_force=reference_bracket_force,
        F_guide_design=F_guide_design,
        F_roller_design=F_roller_design,
        guide_force_per_support=guide_force_per_support,
        profile_names=profile_names,
        profile_mass_per_m=profile_mass_per_m,
        profile_section_modulus=profile_section_modulus,
        profile_sigma=profile_sigma,
        profile_util=profile_util,
        profile_ok=profile_ok,
        selected_idx=selected_idx,
        selected_name=str(profile_names[selected_idx]),
        selected_util=float(profile_util[selected_idx]),
        selected_sigma=float(profile_sigma[selected_idx]),
        selected_mass_per_m=float(profile_mass_per_m[selected_idx]),
    )


R_Ax_local = vars_total["R_Ax"]
C_x_local = vars_total["C_x"]
frame_horizontal_total = F_frame_total[:, 0]
mast_design = evaluate_mast_guide_design(R_Ax_local, F_drive_s_total, frame_horizontal_total)

mast_couple_moment = mast_design["mast_couple"]
mast_couple_index = mast_design["idx_peak"]
mast_couple_moment_peak = mast_design["M_peak"]
mast_couple_peak_s = mast_design["s_peak"]
slider_side_reaction_peak = mast_design["side_peak"]
frame_horizontal_peak = mast_design["frame_x_peak"]
drive_force_peak_per_mechanism = mast_design["drive_peak"]
guide_to_drive_force_ratio = mast_design["ratio"]
mast_bracket_force_est = mast_design["bracket_force"]
mast_reference_bracket_force = mast_design["reference_bracket_force"]
F_guide_design = mast_design["F_guide_design"]
F_roller_design = mast_design["F_roller_design"]
guide_force_per_support_candidates = mast_design["guide_force_per_support"]
mast_profile_names = mast_design["profile_names"]
mast_profile_mass_per_m = mast_design["profile_mass_per_m"]
mast_profile_section_modulus = mast_design["profile_section_modulus"]
mast_profile_sigma = mast_design["profile_sigma"]
mast_profile_utilization = mast_design["profile_util"]
mast_profile_ok = mast_design["profile_ok"]
selected_mast_profile = mast_design["selected_name"]
selected_mast_profile_utilization = mast_design["selected_util"]
selected_mast_profile_sigma = mast_design["selected_sigma"]
selected_mast_profile_mass_per_m = mast_design["selected_mass_per_m"]

fig_mast, ax_mast = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
fig_mast.suptitle("Mast- en schuiverbelasting")

ax_mast[0, 0].plot(t, R_Ax_local, label="A_x schuiver")
ax_mast[0, 0].plot(t, C_x_local, label="C_x frame")
ax_mast[0, 0].plot(t, frame_horizontal_total, label="netto frame x")
ax_mast[0, 0].axhline(0.0, color="black", lw=0.8)
ax_mast[0, 0].set_title("Horizontale reacties")
ax_mast[0, 0].set_xlabel("t [s]"); ax_mast[0, 0].set_ylabel("N")
ax_mast[0, 0].grid(True); ax_mast[0, 0].legend()

ax_mast[0, 1].plot(t, mast_couple_moment)
ax_mast[0, 1].scatter(t[mast_couple_index], mast_couple_moment_peak, color="tab:red", zorder=3)
ax_mast[0, 1].set_title("Indicatief krachtkoppel mast")
ax_mast[0, 1].set_xlabel("t [s]"); ax_mast[0, 1].set_ylabel("|A_x| s [Nm]")
ax_mast[0, 1].grid(True)

ax_mast[1, 0].plot(t, np.abs(R_Ax_local), label="|A_x| geleiding")
ax_mast[1, 0].plot(t, np.abs(F_drive_s_total), label="|F_s| riem/motor")
ax_mast[1, 0].set_title("Geleidingskracht versus aandrijfkracht")
ax_mast[1, 0].set_xlabel("t [s]"); ax_mast[1, 0].set_ylabel("N")
ax_mast[1, 0].grid(True); ax_mast[1, 0].legend()

ax_mast[1, 1].plot(mast_bracket_spacing_candidates, mast_bracket_force_est, "o-")
ax_mast[1, 1].axvline(mast_reference_bracket_spacing, color="black", ls=":", label="referentie")
ax_mast[1, 1].set_title("Indicatieve kracht per muur-/mastbeugel")
ax_mast[1, 1].set_xlabel("verticale beugelafstand [m]"); ax_mast[1, 1].set_ylabel("N per steunpunt")
ax_mast[1, 1].grid(True); ax_mast[1, 1].legend()
plt.show()

fig_guide, ax_guide = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
fig_guide.suptitle("Eerste dimensionering schuivergeleiding en mast")
colors = ["tab:green" if ok else "tab:red" for ok in mast_profile_ok]
ax_guide[0].bar(mast_profile_names, mast_profile_utilization, color=colors)
ax_guide[0].axhline(1.0, color="tab:red", ls="--", label="limiet")
ax_guide[0].set_ylabel("benutting [-]")
ax_guide[0].set_title("Mastprofiel op indicatief moment")
ax_guide[0].tick_params(axis="x", rotation=20)
ax_guide[0].grid(True, axis="y"); ax_guide[0].legend()

ax_guide[1].plot(guide_load_sharing_candidates, guide_force_per_support_candidates, "o-", label="ontwerpbelasting")
ax_guide[1].axvline(loaded_roller_count, color="black", ls=":", label="gekozen aantal")
ax_guide[1].axhline(guide_reference_roller_capacity, color="tab:orange", ls="--", label="referentie rol/glijblok")
ax_guide[1].set_xlabel("aantal dragende rollen/glijblokken")
ax_guide[1].set_ylabel("N per steunpunt")
ax_guide[1].set_title("Schuiver/collar lastverdeling")
ax_guide[1].grid(True); ax_guide[1].legend()
plt.show()

print("Mast- en schuiverbelasting:")
print(f"max lokale zijreactie schuiver A_x       : {slider_side_reaction_peak:.2f} N")
print(f"max |A_x| / max |F_s|                    : {guide_to_drive_force_ratio:.1f}")
print(f"ontwerpbelasting geleiding, SF={guide_safety_factor:.1f}       : {F_guide_design:.2f} N")
print(f"ontwerpbelasting per rol/glijblok ({loaded_roller_count} stuks): {F_roller_design:.2f} N")
print(f"max indicatief mastmoment |A_x|*s        : {mast_couple_moment_peak:.2f} Nm bij s = {mast_couple_peak_s:.3f} m")
print(f"geselecteerde mastquick-check            : {selected_mast_profile}, benutting {selected_mast_profile_utilization:.3f}")
print(f"beugelkracht bij {mast_reference_bracket_spacing:.2f} m afstand        : {mast_reference_bracket_force:.2f} N per steunpunt")
print("Interpretatie: de riem/motor wordt op F_s gedimensioneerd; de schuivergeleiding en mast op A_x, rolcontact en het krachtkoppel.")


## Opslag baseline

De baseline zonder trekveren wordt opgeslagen als `notebook3_overdekking_results.npz`. Dit bestand gebruikt dezelfde hoofdkeys als Notebook 3 en bevat extra keys voor breedte, voorbalk, doekmassa en doorbuiging.


In [ ]:
results_path = Path("notebook3_overdekking_results.npz").resolve()
np.savez(
    results_path,
    t=t, s=s, ds=ds, dds=dds,
    F_drive_s_inertia=F_drive_s_inertia,
    F_drive_s_gravity=F_drive_s_gravity,
    F_drive_s_total=F_drive_s_total,
    F_gravity_component=F_gravity_component,
    F_friction_component=F_friction_component,
    F_slider_drive_component=F_slider_drive_component,
    F_pin_friction_component=F_pin_friction_component,
    F_slider_friction_s=case_total["F_slider_friction_s"],
    N_slider=case_total["N_slider"],
    joint_friction_moments=case_total["joint_friction_moments"],
    joint_normal_forces=case_total["joint_normal_forces"],
    joint_power_loss=case_total["joint_power_loss"],
    pin_joint_names=np.array(pin_joint_names),
    R_Ax_total=vars_total["R_Ax"],
    C_x_total=vars_total["C_x"],
    C_y_total=vars_total["C_y"],
    M_A_total=vars_total["M_A"],
    F_A_total=F_A_total,
    F_C_total=F_C_total,
    F_frame_total=F_frame_total,
    F_A_total_norm=F_A_total_norm,
    F_C_total_norm=F_C_total_norm,
    F_frame_total_norm=F_frame_total_norm,
    slider_side_reaction_peak=slider_side_reaction_peak,
    mast_couple_moment=mast_couple_moment,
    mast_couple_moment_peak=mast_couple_moment_peak,
    mast_couple_peak_s=mast_couple_peak_s,
    mast_frame_horizontal_peak=frame_horizontal_peak,
    mast_bracket_spacing_candidates=mast_bracket_spacing_candidates,
    mast_bracket_force_est=mast_bracket_force_est,
    mast_reference_bracket_spacing=mast_reference_bracket_spacing,
    mast_reference_bracket_force=mast_reference_bracket_force,
    guide_to_drive_force_ratio=guide_to_drive_force_ratio,
    guide_safety_factor=guide_safety_factor,
    loaded_roller_count=loaded_roller_count,
    guide_load_sharing_candidates=guide_load_sharing_candidates,
    guide_reference_roller_capacity=guide_reference_roller_capacity,
    F_guide_design=F_guide_design,
    F_roller_design=F_roller_design,
    guide_force_per_support_candidates=guide_force_per_support_candidates,
    mast_material_name=np.array(mast_material_name),
    mast_allowable_stress=mast_allowable_stress,
    mast_profile_names=mast_profile_names,
    mast_profile_mass_per_m=mast_profile_mass_per_m,
    mast_profile_section_modulus=mast_profile_section_modulus,
    mast_profile_sigma=mast_profile_sigma,
    mast_profile_utilization=mast_profile_utilization,
    mast_profile_ok=mast_profile_ok,
    selected_mast_profile=np.array(selected_mast_profile),
    selected_mast_profile_utilization=selected_mast_profile_utilization,
    selected_mast_profile_sigma=selected_mast_profile_sigma,
    selected_mast_profile_mass_per_m=selected_mast_profile_mass_per_m,
    P_act_total=P_act_total,
    E_act_total=E_act_total,
    T_motor_total=T_motor_total,
    motor_speed_rpm=motor_speed_rpm,
    hold_s_values=hold_s_values,
    F_hold_s=F_hold_s,
    R_Ax_hold=R_Ax_hold,
    static_slider_capacity=static_slider_capacity,
    hold_s_curve=hold_s_curve,
    F_hold_s_curve=F_hold_s_curve,
    R_Ax_hold_curve=R_Ax_hold_curve,
    static_slider_capacity_curve=static_slider_capacity_curve,
    T_hold_motor=T_hold_motor,
    T_hold_lock_required=T_hold_lock_required,
    T_hold_motor_curve=T_hold_motor_curve,
    T_hold_lock_required_curve=T_hold_lock_required_curve,
    # Arbeids-surplus
    A_max_full=A_max_full,
    A_theta_full=A_theta_full,
    P_avg_full=P_avg_full,
    P_peak_full=P_peak_full,
    P_rms_full=P_rms_full,
    F_peak_full=F_peak_full,
    F_rms_full=F_rms_full,
    T_motor_peak_sizing=T_motor_peak_sizing,
    T_motor_rms_sizing=T_motor_rms_sizing,
    # Energiebalans-validatie
    energy_balance_error=energy_balance_error,
    Ekin=Ekin,

    # Bidirectionele motorloadcase
    direction_names=direction_names,
    input_motion_label=np.array(input_motion_label),
    input_is_opening=input_is_opening,
    motor_design_direction=np.array(motor_design_direction),
    t_open=t_open_profile,
    s_open_profile=s_open_profile,
    ds_open_profile=ds_open_profile,
    dds_open_profile=dds_open_profile,
    t_close=t_close_profile,
    s_close_profile=s_close_profile,
    ds_close_profile=ds_close_profile,
    dds_close_profile=dds_close_profile,
    F_drive_s_total_open=F_drive_s_total_open,
    F_drive_s_total_open_position_order=F_drive_s_total_open_position,
    F_drive_s_total_close=F_drive_s_total_close,
    F_drive_s_total_close_position_order=F_drive_s_total_close_position,
    F_drive_s_inertia_open=F_drive_s_inertia_open,
    F_drive_s_inertia_close=F_drive_s_inertia_close,
    P_act_total_open=P_act_total_open,
    P_act_total_close=P_act_total_close,
    E_act_total_open=E_act_total_open,
    E_act_total_close=E_act_total_close,
    E_positive_open=E_positive_open,
    E_positive_close=E_positive_close,
    E_negative_open=E_negative_open,
    E_negative_close=E_negative_close,
    E_net_open=E_net_open,
    E_net_close=E_net_close,
    direction_force_peaks=direction_force_peaks,
    direction_power_peaks=direction_power_peaks,
    direction_positive_energy=direction_positive_energy,
    direction_net_energy=direction_net_energy,
    # Massa- en modelparameters
    masses=np.array([masses[i] for i in range(2, 9)]),
    inertias=np.array([inertias[i] for i in range(2, 9)]),
    payload_mass_K=payload_mass_K,
    line_mass_density=line_mass_density,
    slider_mass=slider_mass,
    rod_outer_diameter=rod_outer_diameter,
    rod_wall_thickness=rod_wall_thickness,
    rod_inner_diameter=rod_inner_diameter,
    rod_material_density=rod_material_density,
    rod_tube_area=rod_tube_area,
    rod_tube_mass_per_m=rod_tube_mass_per_m,
    rod_fittings_line_mass_allowance=rod_fittings_line_mass_allowance,
    total_model_mass=total_model_mass,
    total_system_model_mass=total_system_model_mass,
    payload_mass_K_equivalent=payload_mass_K_equivalent,
    g=g,
    mu_slider=mu_slider,
    mu_slider_static=mu_slider_static,
    c_slider=c_slider,
    mu_pin=mu_pin,
    pin_radius=pin_radius,
    actuator_efficiency=actuator_efficiency,
    actuator_safety_factor=actuator_safety_factor,
    drive_pulley_radius_nb3=drive_pulley_radius_nb3,
    drive_travel_per_rev_nb3=drive_travel_per_rev_nb3,
    pulley_radius_nb3=pulley_radius_nb3,
    screw_lead=screw_lead,
    total_weight=total_weight,
    total_system_weight=total_system_model_mass * g,
    load_case=np.array("overdekking"),
    canopy_width=canopy_width,
    canopy_depth=canopy_depth,
    canopy_area=canopy_area,
    mechanism_count_total=mechanism_count_total,
    mechanism_spacing=mechanism_spacing,
    support_z_positions=support_z_positions,
    front_beam_profile=np.array(front_beam_profile),
    front_beam_mass_per_m=front_beam_mass_per_m,
    front_beam_mass_total=front_beam_mass_total,
    fabric_areal_density=fabric_areal_density,
    fabric_mass_total=fabric_mass_total,
    fabric_mass_to_K_fraction=fabric_mass_to_K_fraction,
    fittings_mass_per_K=fittings_mass_per_K,
    beam_area=beam_area,
    beam_I=beam_I,
    beam_span=beam_span,
    beam_line_load_mass=beam_line_load_mass,
    beam_line_load_force=beam_line_load_force,
    beam_deflection_max=beam_deflection_max,
    beam_deflection_limit_L300=beam_deflection_limit_L300,
    aluminium_E=aluminium_E,
    aluminium_density=aluminium_density,
    # Structurele weercontrole voorbalk
    weather_case_names=weather_case_names,
    weather_area_pressure_cases=weather_area_pressure_cases,
    beam_q_line_cases=beam_q_line_cases,
    beam_V_max_cases=beam_V_max_cases,
    beam_M_max_cases=beam_M_max_cases,
    beam_deflection_cases=beam_deflection_cases,
    beam_T_max_cases=beam_T_max_cases,
    beam_twist_cases=beam_twist_cases,
    beam_sigma_bending_cases=beam_sigma_bending_cases,
    beam_tau_shear_cases=beam_tau_shear_cases,
    beam_tau_torsion_cases=beam_tau_torsion_cases,
    beam_von_mises_cases=beam_von_mises_cases,
    beam_utilization_deflection=beam_utilization_deflection,
    beam_utilization_stress=beam_utilization_stress,
    beam_utilization_torsion=beam_utilization_torsion,
    beam_utilization_max=beam_utilization_max,
    beam_governing_case=np.array(beam_governing_case),
    beam_structural_ok=beam_structural_ok,
    beam_structural_status=np.array(beam_structural_status),
    beam_I_strong=beam_I_strong,
    beam_I_weak=beam_I_weak,
    beam_section_modulus_strong=beam_section_modulus_strong,
    beam_section_modulus_weak=beam_section_modulus_weak,
    beam_shear_area=beam_shear_area,
    beam_torsion_constant=beam_torsion_constant,
    beam_G=beam_G,
    allowable_deflection=allowable_deflection,
    allowable_stress=allowable_stress,
    allowable_twist_rad=allowable_twist_rad,
    wind_basic_velocity=wind_basic_velocity,
    wind_peak_pressure=wind_peak_pressure,
    wind_down_pressure=wind_down_pressure,
    wind_uplift_pressure=wind_uplift_pressure,
    snow_pressure=snow_pressure,
    front_beam_tributary_depth_fraction=front_beam_tributary_depth_fraction,
    front_beam_load_eccentricity=front_beam_load_eccentricity,
    profile_screen_names=profile_screen_names,
    profile_screen_mass_per_m=profile_screen_mass_per_m,
    profile_screen_payload_mass_K=profile_screen_payload_mass_K,
    profile_screen_max_util=profile_screen_max_util,
    profile_screen_governing_case=profile_screen_governing_case.astype(str),
    profile_screen_ok=profile_screen_ok,
    profile_screen_max_deflection=profile_screen_max_deflection,
    profile_screen_max_von_mises=profile_screen_max_von_mises,
    profile_screen_max_twist=profile_screen_max_twist,
    dyn_residual_total=case_total["residual"],
    dyn_cond_total=case_total["cond"],
    inertia_check_residual=inertia_check_residual,
    inertia_reference_note=np.array("eigen overdekkingsmassa; geen directe Notebook-2-vergelijking"),
    max_diff_notebook2=max_diff_notebook2,
)

print("Overdekking-baseline opgeslagen in:")
print(results_path)
print()
print("SAMENVATTING - OVERDEKKING BASELINE")
print("=" * 56)
print(f"max |F_s| inertie                    : {np.max(np.abs(F_drive_s_inertia)):.2f} N")
print(f"max |F_s| met zwaartekracht          : {np.max(np.abs(F_drive_s_gravity)):.2f} N")
print(f"max |F_s| totaal                     : {np.max(np.abs(F_drive_s_total)):.2f} N")
print(f"open stand |F_hold|                  : {abs(F_hold_s_curve[open_idx_curve]):.2f} N")
print(f"piek actuatorvermogen                : {P_positive_peak:.2f} W")
print(f"piek poeliekoppel uitgang            : {T_motor_peak:.4f} Nm")
print(f"max steunreactie schuiver/actuator   : {np.max(F_A_total_norm):.2f} N")
print(f"max steunreactie C                   : {np.max(F_C_total_norm):.2f} N")
print(f"max netto framekracht                : {np.max(F_frame_total_norm):.2f} N")
print(f"arbeids-surplus A_max        : {A_max_full:.4f} J")
print(f"max energiebalans-fout               : {np.max(np.abs(energy_balance_error)):.3e} W")
print(f"payload_mass_K equivalent            : {payload_mass_K:.2f} kg per mechanisme")
print(f"voorbalkdoorbuiging                  : {1000*beam_deflection_max:.1f} mm")
print(f"totale aandrijfkracht alle mechanismen: {mechanism_count_total*np.max(np.abs(F_drive_s_total[active_motion_mask])):.2f} N")


## Optionele trekveren

De trekveren worden per mechanisme gemodelleerd als gewone directe trekveren naast de schuiver/collar. Er is dus geen extra kabel-, poelie- of hefboomverhouding in het veermodel: de veerrek volgt dezelfde slag als de schuiver.

De dynamica gebruikt de resulterende verticale hulpkracht op de schuiver. De standaard veerconstante is bewust laag (`0.010 N/mm` per veer), omdat de schuiverslag groot is. Een stijvere compacte veer zou in gesloten stand te veel kracht leveren en het mechanisme overcompenseren. De automatische dimensionering kiest de voorspanning/open-kracht op basis van de baseline-houdkracht, met een limiet zodat de motor en rem controle houden.


In [ ]:
spring_results_written = False
spring_force_up_total = np.zeros_like(t)
spring_force_s_total = np.zeros_like(t)
spring_energy_stored = np.zeros_like(t)
spring_energy_delta = 0.0

if compute_spring_assist_case:
    F_drive_s_baseline = F_drive_s_total.copy()
    F_hold_s_baseline_curve = F_hold_s_curve.copy()
    upward_hold_required_curve = np.maximum(-F_hold_s_baseline_curve, 0.0)
    open_idx_curve = int(np.argmin(hold_s_curve))
    closed_idx_curve = int(np.argmax(hold_s_curve))

    if spring_design_mode == "fraction_of_baseline_hold":
        spring_force_open_goal = spring_assist_fraction_open * upward_hold_required_curve[open_idx_curve]
        spring_force_closed_goal = spring_assist_fraction_closed * upward_hold_required_curve[closed_idx_curve]
        if spring_force_closed_goal < spring_force_open_goal:
            spring_force_closed_goal = spring_force_open_goal

        spring_k_per_spring = 1000.0 * spring_direct_rate_per_spring_N_per_mm  # [N/m]
        spring_k_total = spring_count_per_mechanism * spring_k_per_spring
        spring_force_delta_total = spring_k_total * stroke
        spring_force_open_target = min(spring_force_open_goal, spring_force_closed_goal - spring_force_delta_total)
        spring_force_open_target = max(0.0, spring_force_open_target)
    else:
        spring_force_open_target = float(spring_force_open_total_manual)
        spring_force_closed_target = float(spring_force_closed_total_manual)
        if spring_force_closed_target < spring_force_open_target:
            raise ValueError("Voor deze verticale trekveer moet de gesloten veerkracht minstens gelijk zijn aan de open veerkracht.")
        spring_k_total = (spring_force_closed_target - spring_force_open_target) / stroke
        spring_k_per_spring = spring_k_total / spring_count_per_mechanism
        spring_direct_rate_per_spring_N_per_mm = spring_k_per_spring / 1000.0
        spring_physical_rate_per_spring_N_per_mm = spring_direct_rate_per_spring_N_per_mm

    assist_cap_curve = spring_max_assist_fraction * upward_hold_required_curve
    cap_open_curve = assist_cap_curve - spring_k_total * (hold_s_curve - s_open)
    max_open_force_by_cap = float(np.min(cap_open_curve))
    if max_open_force_by_cap < -1e-9:
        raise ValueError("De gekozen directe veerconstante is te hoog: zelfs zonder voorspanning overschrijdt ze de assistlimiet.")
    spring_force_open_total = min(spring_force_open_target, max(0.0, max_open_force_by_cap))
    spring_force_closed_total = spring_force_open_total + spring_k_total * stroke
    spring_force_open_per_spring = spring_force_open_total / spring_count_per_mechanism
    spring_force_closed_per_spring = spring_force_closed_total / spring_count_per_mechanism
    spring_scale_factor = spring_force_open_total / spring_force_open_target if spring_force_open_target > 1e-12 else 1.0

    spring_physical_rate_per_spring = spring_k_per_spring
    spring_physical_rate_per_spring_N_per_mm = spring_k_per_spring / 1000.0
    spring_motion_ratio = 1.0
    spring_initial_tension_per_spring = spring_force_open_per_spring
    spring_force_open_physical_per_spring = spring_force_open_per_spring
    spring_force_closed_physical_per_spring = spring_force_closed_per_spring
    spring_physical_extension_closed = stroke
    spring_physical_preload_extension = 0.0

    spring_force_up_total = np.maximum(spring_force_open_total + spring_k_total * (s - s_open), 0.0)
    spring_force_s_total = -spring_force_up_total
    spring_energy_stored = spring_force_open_total * (s - s_open) + 0.5 * spring_k_total * (s - s_open) ** 2
    spring_energy_delta = float(spring_force_open_total * stroke + 0.5 * spring_k_total * stroke ** 2)
    spring_physical_extension = s - s_open
    spring_force_physical_per_spring = spring_force_up_total / spring_count_per_mechanism
    spring_force_physical_total = spring_force_up_total
    spring_physical_extension_hold_curve = hold_s_curve - s_open
    spring_force_physical_hold_curve_per_spring = np.maximum(spring_force_open_total + spring_k_total * (hold_s_curve - s_open), 0.0) / spring_count_per_mechanism

    case_gravity_spring = solve_case(include_gravity=True, include_friction=False, include_spring=True, spring_force_up_time=spring_force_up_total)
    case_total_spring = solve_case(include_gravity=True, include_friction=True, include_spring=True, spring_force_up_time=spring_force_up_total, velocity_sign=1.0)
    case_total_spring_close = solve_case(include_gravity=True, include_friction=True, include_spring=True, spring_force_up_time=spring_force_up_total, velocity_sign=-1.0)
    vars_gravity_spring = unpack_case(case_gravity_spring)
    vars_total_spring = unpack_case(case_total_spring)
    vars_total_spring_close = unpack_case(case_total_spring_close)

    F_drive_s_gravity_spring = -vars_gravity_spring["F_act_y"]
    F_drive_s_total_spring = -vars_total_spring["F_act_y"]
    F_drive_s_total_spring_opposite_position = -vars_total_spring_close["F_act_y"]
    if input_is_opening:
        F_drive_s_total_spring_open = F_drive_s_total_spring.copy()
        F_drive_s_total_spring_open_position = F_drive_s_total_spring.copy()
        F_drive_s_total_spring_close = F_drive_s_total_spring_opposite_position[::-1]
        F_drive_s_total_spring_close_position = F_drive_s_total_spring_opposite_position.copy()
    else:
        F_drive_s_total_spring_close = F_drive_s_total_spring.copy()
        F_drive_s_total_spring_close_position = F_drive_s_total_spring.copy()
        F_drive_s_total_spring_open = F_drive_s_total_spring_opposite_position[::-1]
        F_drive_s_total_spring_open_position = F_drive_s_total_spring_opposite_position.copy()
    F_gravity_component_spring = F_drive_s_gravity_spring - F_drive_s_inertia
    F_spring_assist_component = F_drive_s_gravity_spring - F_drive_s_gravity
    F_friction_component_spring = F_drive_s_total_spring - F_drive_s_gravity_spring
    F_friction_component_spring_close_position = F_drive_s_total_spring_close_position - F_drive_s_gravity_spring
    F_slider_drive_component_spring = -case_total_spring["F_slider_friction_s"]
    F_slider_drive_component_spring_close_position = -case_total_spring_close["F_slider_friction_s"]
    F_pin_friction_component_spring = F_friction_component_spring - F_slider_drive_component_spring
    F_pin_friction_component_spring_close_position = F_friction_component_spring_close_position - F_slider_drive_component_spring_close_position

    F_hold_s_curve_spring = np.zeros_like(hold_s_curve)
    R_Ax_hold_curve_spring = np.zeros_like(hold_s_curve)
    C_x_hold_curve_spring = np.zeros_like(hold_s_curve)
    C_y_hold_curve_spring = np.zeros_like(hold_s_curve)
    hold_curve_indices = np.argsort(s)
    for out_i, k in enumerate(hold_curve_indices):
        A_hold, b_hold = build_inverse_dynamics_system(k, include_gravity=True, include_spring=True, spring_force_up=float(spring_force_up_total[k]), static=True)
        w_hold = np.linalg.solve(A_hold, b_hold)
        F_hold_s_curve_spring[out_i] = -w_hold[unknown_index["F_act_y"]]
        R_Ax_hold_curve_spring[out_i] = w_hold[unknown_index["R_Ax"]]
        C_x_hold_curve_spring[out_i] = w_hold[unknown_index["C_x"]]
        C_y_hold_curve_spring[out_i] = w_hold[unknown_index["C_y"]]
    static_slider_capacity_curve_spring = mu_slider_static * np.abs(R_Ax_hold_curve_spring)
    T_hold_motor_curve_spring = F_hold_s_curve_spring * drive_travel_per_rev_nb3 / (2 * np.pi * actuator_efficiency)
    T_hold_lock_required_curve_spring = actuator_safety_factor * np.abs(F_hold_s_curve_spring) * drive_travel_per_rev_nb3 / (2 * np.pi * actuator_efficiency)
    sample_targets = np.linspace(np.min(s), np.max(s), 5)
    sample_indices_curve = np.array([int(np.argmin(np.abs(hold_s_curve - target))) for target in sample_targets])
    hold_s_values_spring = hold_s_curve[sample_indices_curve]
    F_hold_s_spring = F_hold_s_curve_spring[sample_indices_curve]
    R_Ax_hold_spring = R_Ax_hold_curve_spring[sample_indices_curve]
    static_slider_capacity_spring = static_slider_capacity_curve_spring[sample_indices_curve]
    T_hold_motor_spring = T_hold_motor_curve_spring[sample_indices_curve]
    T_hold_lock_required_spring = T_hold_lock_required_curve_spring[sample_indices_curve]

    P_current_spring_position, P_current_spring_time, E_current_spring_time, E_positive_current_spring, E_negative_current_spring, E_net_current_spring = direction_energy_arrays(
        F_drive_s_total_spring, ds, reverse_for_time=False
    )
    P_opposite_spring_position, P_opposite_spring_time, E_opposite_spring_time, E_positive_opposite_spring, E_negative_opposite_spring, E_net_opposite_spring = direction_energy_arrays(
        F_drive_s_total_spring_opposite_position, opposite_ds_position, reverse_for_time=True
    )
    if input_is_opening:
        P_act_total_spring_position = P_current_spring_position
        P_act_total_spring = P_current_spring_time
        E_act_total_spring = E_current_spring_time
        E_positive_open_spring = E_positive_current_spring
        E_negative_open_spring = E_negative_current_spring
        E_net_open_spring = E_net_current_spring
        P_act_total_spring_close_position = P_opposite_spring_position
        P_act_total_spring_close = P_opposite_spring_time
        E_act_total_spring_close = E_opposite_spring_time
        E_positive_close_spring = E_positive_opposite_spring
        E_negative_close_spring = E_negative_opposite_spring
        E_net_close_spring = E_net_opposite_spring
    else:
        P_act_total_spring_close_position = P_current_spring_position
        P_act_total_spring_close = P_current_spring_time
        E_act_total_spring_close = E_current_spring_time
        E_positive_close_spring = E_positive_current_spring
        E_negative_close_spring = E_negative_current_spring
        E_net_close_spring = E_net_current_spring
        P_act_total_spring_position = P_opposite_spring_position
        P_act_total_spring = P_opposite_spring_time
        E_act_total_spring = E_opposite_spring_time
        E_positive_open_spring = E_positive_opposite_spring
        E_negative_open_spring = E_negative_opposite_spring
        E_net_open_spring = E_net_opposite_spring
    T_motor_total_spring = F_drive_s_total_spring * drive_travel_per_rev_nb3 / (2 * np.pi * actuator_efficiency)
    motor_speed_rpm_spring = 60.0 * ds / drive_travel_per_rev_nb3
    force_peak_open_spring = float(np.max(np.abs(F_drive_s_total_spring_open[motion_mask_open])))
    force_peak_close_spring = float(np.max(np.abs(F_drive_s_total_spring_close[motion_mask_close])))
    power_peak_open_spring = float(np.max(np.maximum(P_act_total_spring, 0.0)))
    power_peak_close_spring = float(np.max(np.maximum(P_act_total_spring_close, 0.0)))
    motor_design_direction_spring = "openen" if max(force_peak_open_spring, power_peak_open_spring) >= max(force_peak_close_spring, power_peak_close_spring) else "sluiten"
    direction_force_peaks_spring = np.array([force_peak_open_spring, force_peak_close_spring])
    direction_power_peaks_spring = np.array([power_peak_open_spring, power_peak_close_spring])
    direction_positive_energy_spring = np.array([E_positive_open_spring, E_positive_close_spring])
    direction_net_energy_spring = np.array([E_net_open_spring, E_net_close_spring])
    P_load_full_spring = P_act_total_spring
    P_avg_full_spring = P_load_full_spring.mean()
    P_peak_full_spring = np.abs(P_load_full_spring).max()
    P_rms_full_spring = np.sqrt(np.mean(P_load_full_spring**2))
    F_peak_full_spring = np.abs(F_drive_s_total_spring_open[motion_mask_open]).max()
    F_rms_full_spring = np.sqrt(np.mean(F_drive_s_total_spring_open[motion_mask_open]**2))
    A_theta_full_spring = np.cumsum((P_load_full_spring - P_avg_full_spring) * Ts)
    A_max_full_spring = A_theta_full_spring.max() - A_theta_full_spring.min()
    T_motor_peak_sizing_spring = actuator_safety_factor * F_peak_full_spring * drive_travel_per_rev_nb3 / (2 * np.pi * actuator_efficiency)
    T_motor_rms_sizing_spring = actuator_safety_factor * F_rms_full_spring * drive_travel_per_rev_nb3 / (2 * np.pi * actuator_efficiency)

    Ekin_spring = Ekin.copy()
    P_spring_storage = spring_force_up_total * ds
    P_friction_spring = -case_total_spring["F_slider_friction_s"] * ds + np.sum(case_total_spring["joint_power_loss"], axis=1)
    # P_gravity_val en dEkin_dt komen uit de baseline-energiebalanscel en blijven geldig voor dezelfde massa/kinematica.
    energy_balance_error_spring = P_act_total_spring - (dEkin_dt + P_gravity_val + P_friction_spring + P_spring_storage)

    F_slider_global_spring = np.column_stack((np.zeros(n_steps), -case_total_spring["F_slider_friction_s"]))
    F_A_total_spring = np.column_stack((vars_total_spring["R_Ax"], vars_total_spring["F_act_y"])) + F_slider_global_spring
    F_C_total_spring = np.column_stack((vars_total_spring["C_x"], vars_total_spring["C_y"]))
    F_spring_anchor = np.column_stack((np.zeros(n_steps), -spring_force_up_total))
    F_spring_physical_anchor = np.column_stack((np.zeros(n_steps), -spring_force_physical_total))
    F_frame_total_spring = F_A_total_spring + F_C_total_spring + F_spring_anchor
    F_A_total_norm_spring = np.linalg.norm(F_A_total_spring, axis=1)
    F_C_total_norm_spring = np.linalg.norm(F_C_total_spring, axis=1)
    F_frame_total_norm_spring = np.linalg.norm(F_frame_total_spring, axis=1)

    mast_design_spring = evaluate_mast_guide_design(vars_total_spring["R_Ax"], F_drive_s_total_spring, F_frame_total_spring[:, 0])
    mast_couple_moment_spring = mast_design_spring["mast_couple"]
    mast_couple_moment_peak_spring = mast_design_spring["M_peak"]
    mast_couple_peak_s_spring = mast_design_spring["s_peak"]
    slider_side_reaction_peak_spring = mast_design_spring["side_peak"]
    frame_horizontal_peak_spring = mast_design_spring["frame_x_peak"]
    guide_to_drive_force_ratio_spring = mast_design_spring["ratio"]
    mast_bracket_force_est_spring = mast_design_spring["bracket_force"]
    mast_reference_bracket_force_spring = mast_design_spring["reference_bracket_force"]
    F_guide_design_spring = mast_design_spring["F_guide_design"]
    F_roller_design_spring = mast_design_spring["F_roller_design"]
    guide_force_per_support_candidates_spring = mast_design_spring["guide_force_per_support"]
    mast_profile_sigma_spring = mast_design_spring["profile_sigma"]
    mast_profile_utilization_spring = mast_design_spring["profile_util"]
    mast_profile_ok_spring = mast_design_spring["profile_ok"]
    selected_mast_profile_spring = mast_design_spring["selected_name"]
    selected_mast_profile_utilization_spring = mast_design_spring["selected_util"]
    selected_mast_profile_sigma_spring = mast_design_spring["selected_sigma"]
    selected_mast_profile_mass_per_m_spring = mast_design_spring["selected_mass_per_m"]

    baseline_open_peak = float(np.max(np.abs(F_drive_s_total_open[motion_mask_open])))
    spring_open_peak = float(np.max(np.abs(F_drive_s_total_spring_open[motion_mask_open])))
    opening_peak_reduction = baseline_open_peak - spring_open_peak
    opening_peak_reduction_pct = 100.0 * opening_peak_reduction / baseline_open_peak if baseline_open_peak > 1e-9 else 0.0

    fig_spring, ax_spring = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
    fig_spring.suptitle("Effect van trekveren - per mechanisme")
    ax_spring[0, 0].plot(hold_s_curve, spring_force_open_total + spring_k_total * (hold_s_curve - s_open), label="equiv. schuiverkracht omhoog")
    ax_spring[0, 0].plot(hold_s_curve, spring_force_physical_hold_curve_per_spring, ":", label="directe veerkracht per veer")
    ax_spring[0, 0].plot(s, spring_energy_stored, label="opgeslagen veerenergie")
    ax_spring[0, 0].set_xlabel("s [m]"); ax_spring[0, 0].set_ylabel("N / J")
    ax_spring[0, 0].set_title("Veerwet"); ax_spring[0, 0].grid(True); ax_spring[0, 0].legend()
    ax_spring[0, 1].plot(s_open_profile, F_drive_s_total_open, label="zonder veer, openen")
    ax_spring[0, 1].plot(s_close_profile, F_drive_s_total_close, "--", label="zonder veer, sluiten")
    ax_spring[0, 1].plot(s_open_profile, F_drive_s_total_spring_open, label="met veer, openen")
    ax_spring[0, 1].plot(s_close_profile, F_drive_s_total_spring_close, "--", label="met veer, sluiten")
    ax_spring[0, 1].invert_xaxis()
    ax_spring[0, 1].set_xlabel("s [m]"); ax_spring[0, 1].set_ylabel("F_s [N]")
    ax_spring[0, 1].set_title("Aandrijfkracht openen/sluiten"); ax_spring[0, 1].grid(True); ax_spring[0, 1].legend()
    ax_spring[1, 0].plot(t_open_profile, P_act_total_open, label="zonder veer, openen")
    ax_spring[1, 0].plot(t_close_profile, P_act_total_close, "--", label="zonder veer, sluiten")
    ax_spring[1, 0].plot(t_open_profile, P_act_total_spring, label="met veer, openen")
    ax_spring[1, 0].plot(t_close_profile, P_act_total_spring_close, "--", label="met veer, sluiten")
    ax_spring[1, 0].axhline(0.0, color="black", linewidth=0.8)
    ax_spring[1, 0].set_xlabel("t [s]"); ax_spring[1, 0].set_ylabel("P [W]")
    ax_spring[1, 0].set_title("Actuatorvermogen"); ax_spring[1, 0].grid(True); ax_spring[1, 0].legend()
    ax_spring[1, 1].plot(hold_s_curve, np.abs(F_hold_s_curve), label="zonder veer")
    ax_spring[1, 1].plot(hold_s_curve, np.abs(F_hold_s_curve_spring), label="met trekveren")
    ax_spring[1, 1].set_xlabel("s [m]"); ax_spring[1, 1].set_ylabel("|F_hold| [N]")
    ax_spring[1, 1].set_title("Houdkracht"); ax_spring[1, 1].grid(True); ax_spring[1, 1].legend()
    plt.show()

    spring_path = Path("notebook3_overdekking_trekveren_results.npz").resolve()
    np.savez(
        spring_path,
        t=t, s=s, ds=ds, dds=dds,
        F_drive_s_inertia=F_drive_s_inertia,
        F_drive_s_gravity=F_drive_s_gravity_spring,
        F_drive_s_total=F_drive_s_total_spring,
        F_gravity_component=F_gravity_component_spring,
        F_friction_component=F_friction_component_spring,
        F_slider_drive_component=F_slider_drive_component_spring,
        F_pin_friction_component=F_pin_friction_component_spring,
        F_slider_friction_s=case_total_spring["F_slider_friction_s"],
        F_slider_friction_s_close_position_order=case_total_spring_close["F_slider_friction_s"],
        N_slider=case_total_spring["N_slider"],
        joint_friction_moments=case_total_spring["joint_friction_moments"],
        joint_normal_forces=case_total_spring["joint_normal_forces"],
        joint_power_loss=case_total_spring["joint_power_loss"],
        pin_joint_names=np.array(pin_joint_names),
        R_Ax_total=vars_total_spring["R_Ax"],
        C_x_total=vars_total_spring["C_x"],
        C_y_total=vars_total_spring["C_y"],
        M_A_total=vars_total_spring["M_A"],
        F_A_total=F_A_total_spring,
        F_C_total=F_C_total_spring,
        F_frame_total=F_frame_total_spring,
        F_A_total_norm=F_A_total_norm_spring,
        F_C_total_norm=F_C_total_norm_spring,
        F_frame_total_norm=F_frame_total_norm_spring,
        slider_side_reaction_peak=slider_side_reaction_peak_spring,
        mast_couple_moment=mast_couple_moment_spring,
        mast_couple_moment_peak=mast_couple_moment_peak_spring,
        mast_couple_peak_s=mast_couple_peak_s_spring,
        mast_frame_horizontal_peak=frame_horizontal_peak_spring,
        mast_bracket_spacing_candidates=mast_bracket_spacing_candidates,
        mast_bracket_force_est=mast_bracket_force_est_spring,
        mast_reference_bracket_spacing=mast_reference_bracket_spacing,
        mast_reference_bracket_force=mast_reference_bracket_force_spring,
        guide_to_drive_force_ratio=guide_to_drive_force_ratio_spring,
        guide_safety_factor=guide_safety_factor,
        loaded_roller_count=loaded_roller_count,
        guide_load_sharing_candidates=guide_load_sharing_candidates,
        guide_reference_roller_capacity=guide_reference_roller_capacity,
        F_guide_design=F_guide_design_spring,
        F_roller_design=F_roller_design_spring,
        guide_force_per_support_candidates=guide_force_per_support_candidates_spring,
        mast_material_name=np.array(mast_material_name),
        mast_allowable_stress=mast_allowable_stress,
        mast_profile_names=mast_profile_names,
        mast_profile_mass_per_m=mast_profile_mass_per_m,
        mast_profile_section_modulus=mast_profile_section_modulus,
        mast_profile_sigma=mast_profile_sigma_spring,
        mast_profile_utilization=mast_profile_utilization_spring,
        mast_profile_ok=mast_profile_ok_spring,
        selected_mast_profile=np.array(selected_mast_profile_spring),
        selected_mast_profile_utilization=selected_mast_profile_utilization_spring,
        selected_mast_profile_sigma=selected_mast_profile_sigma_spring,
        selected_mast_profile_mass_per_m=selected_mast_profile_mass_per_m_spring,
        P_act_total=P_act_total_spring,
        E_act_total=E_act_total_spring,
        T_motor_total=T_motor_total_spring,
        motor_speed_rpm=motor_speed_rpm_spring,

        # Bidirectionele motorloadcase
        direction_names=direction_names,
        input_motion_label=np.array(input_motion_label),
        input_is_opening=input_is_opening,
        motor_design_direction=np.array(motor_design_direction_spring),
        t_open=t_open_profile,
        s_open_profile=s_open_profile,
        ds_open_profile=ds_open_profile,
        dds_open_profile=dds_open_profile,
        t_close=t_close_profile,
        s_close_profile=s_close_profile,
        ds_close_profile=ds_close_profile,
        dds_close_profile=dds_close_profile,
        F_drive_s_total_open=F_drive_s_total_spring_open,
        F_drive_s_total_open_position_order=F_drive_s_total_spring_open_position,
        F_drive_s_total_close=F_drive_s_total_spring_close,
        F_drive_s_total_close_position_order=F_drive_s_total_spring_close_position,
        F_drive_s_inertia_open=F_drive_s_inertia_open,
        F_drive_s_inertia_close=F_drive_s_inertia_close,
        P_act_total_open=P_act_total_spring,
        P_act_total_close=P_act_total_spring_close,
        E_act_total_open=E_act_total_spring,
        E_act_total_close=E_act_total_spring_close,
        E_positive_open=E_positive_open_spring,
        E_positive_close=E_positive_close_spring,
        E_negative_open=E_negative_open_spring,
        E_negative_close=E_negative_close_spring,
        E_net_open=E_net_open_spring,
        E_net_close=E_net_close_spring,
        direction_force_peaks=direction_force_peaks_spring,
        direction_power_peaks=direction_power_peaks_spring,
        direction_positive_energy=direction_positive_energy_spring,
        direction_net_energy=direction_net_energy_spring,
        hold_s_values=hold_s_values_spring,
        F_hold_s=F_hold_s_spring,
        R_Ax_hold=R_Ax_hold_spring,
        static_slider_capacity=static_slider_capacity_spring,
        hold_s_curve=hold_s_curve,
        F_hold_s_curve=F_hold_s_curve_spring,
        R_Ax_hold_curve=R_Ax_hold_curve_spring,
        static_slider_capacity_curve=static_slider_capacity_curve_spring,
        T_hold_motor=T_hold_motor_spring,
        T_hold_lock_required=T_hold_lock_required_spring,
        T_hold_motor_curve=T_hold_motor_curve_spring,
        T_hold_lock_required_curve=T_hold_lock_required_curve_spring,
        A_max_full=A_max_full_spring,
        A_theta_full=A_theta_full_spring,
        P_avg_full=P_avg_full_spring,
        P_peak_full=P_peak_full_spring,
        P_rms_full=P_rms_full_spring,
        F_peak_full=F_peak_full_spring,
        F_rms_full=F_rms_full_spring,
        T_motor_peak_sizing=T_motor_peak_sizing_spring,
        T_motor_rms_sizing=T_motor_rms_sizing_spring,
        energy_balance_error=energy_balance_error_spring,
        Ekin=Ekin_spring,
        masses=np.array([masses[i] for i in range(2, 9)]),
        inertias=np.array([inertias[i] for i in range(2, 9)]),
        payload_mass_K=payload_mass_K,
        payload_mass_K_equivalent=payload_mass_K_equivalent,
        line_mass_density=line_mass_density,
        slider_mass=slider_mass,
        rod_outer_diameter=rod_outer_diameter,
        rod_wall_thickness=rod_wall_thickness,
        rod_inner_diameter=rod_inner_diameter,
        rod_material_density=rod_material_density,
        rod_tube_area=rod_tube_area,
        rod_tube_mass_per_m=rod_tube_mass_per_m,
        rod_fittings_line_mass_allowance=rod_fittings_line_mass_allowance,
        total_model_mass=total_model_mass,
        total_system_model_mass=total_system_model_mass,
        g=g,
        mu_slider=mu_slider,
        mu_slider_static=mu_slider_static,
        c_slider=c_slider,
        mu_pin=mu_pin,
        pin_radius=pin_radius,
        actuator_efficiency=actuator_efficiency,
        actuator_safety_factor=actuator_safety_factor,
        drive_pulley_radius_nb3=drive_pulley_radius_nb3,
        drive_travel_per_rev_nb3=drive_travel_per_rev_nb3,
        pulley_radius_nb3=pulley_radius_nb3,
        screw_lead=screw_lead,
        total_weight=total_weight,
        total_system_weight=total_system_model_mass * g,
        dyn_residual_total=case_total_spring["residual"],
        dyn_residual_total_close=case_total_spring_close["residual"],
        dyn_cond_total=case_total_spring["cond"],
        inertia_check_residual=inertia_check_residual,
        inertia_reference_note=np.array("eigen overdekkingsmassa; geen directe Notebook-2-vergelijking"),
        max_diff_notebook2=max_diff_notebook2,
        load_case=np.array("overdekking_trekveren"),
        canopy_width=canopy_width,
        canopy_depth=canopy_depth,
        canopy_area=canopy_area,
        mechanism_count_total=mechanism_count_total,
        mechanism_spacing=mechanism_spacing,
        support_z_positions=support_z_positions,
        front_beam_profile=np.array(front_beam_profile),
        front_beam_mass_per_m=front_beam_mass_per_m,
        front_beam_mass_total=front_beam_mass_total,
        fabric_areal_density=fabric_areal_density,
        fabric_mass_total=fabric_mass_total,
        fabric_mass_to_K_fraction=fabric_mass_to_K_fraction,
        fittings_mass_per_K=fittings_mass_per_K,
        beam_area=beam_area,
        beam_I=beam_I,
        beam_span=beam_span,
        beam_line_load_mass=beam_line_load_mass,
        beam_line_load_force=beam_line_load_force,
        beam_deflection_max=beam_deflection_max,
        beam_deflection_limit_L300=beam_deflection_limit_L300,
        aluminium_E=aluminium_E,
        aluminium_density=aluminium_density,
        # Structurele weercontrole voorbalk
        weather_case_names=weather_case_names,
        weather_area_pressure_cases=weather_area_pressure_cases,
        beam_q_line_cases=beam_q_line_cases,
        beam_V_max_cases=beam_V_max_cases,
        beam_M_max_cases=beam_M_max_cases,
        beam_deflection_cases=beam_deflection_cases,
        beam_T_max_cases=beam_T_max_cases,
        beam_twist_cases=beam_twist_cases,
        beam_sigma_bending_cases=beam_sigma_bending_cases,
        beam_tau_shear_cases=beam_tau_shear_cases,
        beam_tau_torsion_cases=beam_tau_torsion_cases,
        beam_von_mises_cases=beam_von_mises_cases,
        beam_utilization_deflection=beam_utilization_deflection,
        beam_utilization_stress=beam_utilization_stress,
        beam_utilization_torsion=beam_utilization_torsion,
        beam_utilization_max=beam_utilization_max,
        beam_governing_case=np.array(beam_governing_case),
        beam_structural_ok=beam_structural_ok,
        beam_structural_status=np.array(beam_structural_status),
        beam_I_strong=beam_I_strong,
        beam_I_weak=beam_I_weak,
        beam_section_modulus_strong=beam_section_modulus_strong,
        beam_section_modulus_weak=beam_section_modulus_weak,
        beam_shear_area=beam_shear_area,
        beam_torsion_constant=beam_torsion_constant,
        beam_G=beam_G,
        allowable_deflection=allowable_deflection,
        allowable_stress=allowable_stress,
        allowable_twist_rad=allowable_twist_rad,
        wind_basic_velocity=wind_basic_velocity,
        wind_peak_pressure=wind_peak_pressure,
        wind_down_pressure=wind_down_pressure,
        wind_uplift_pressure=wind_uplift_pressure,
        snow_pressure=snow_pressure,
        front_beam_tributary_depth_fraction=front_beam_tributary_depth_fraction,
        front_beam_load_eccentricity=front_beam_load_eccentricity,
        profile_screen_names=profile_screen_names,
        profile_screen_mass_per_m=profile_screen_mass_per_m,
        profile_screen_payload_mass_K=profile_screen_payload_mass_K,
        profile_screen_max_util=profile_screen_max_util,
        profile_screen_governing_case=profile_screen_governing_case.astype(str),
        profile_screen_ok=profile_screen_ok,
        profile_screen_max_deflection=profile_screen_max_deflection,
        profile_screen_max_von_mises=profile_screen_max_von_mises,
        profile_screen_max_twist=profile_screen_max_twist,
        spring_count=spring_count_per_mechanism,
        spring_force_up_total=spring_force_up_total,
        spring_force_s_total=spring_force_s_total,
        spring_force_open_total=spring_force_open_total,
        spring_force_closed_total=spring_force_closed_total,
        spring_k_total=spring_k_total,
        spring_k_per_spring=spring_k_per_spring,
        spring_force_open_per_spring=spring_force_open_per_spring,
        spring_force_closed_per_spring=spring_force_closed_per_spring,
        spring_physical_rate_per_spring_N_per_mm=spring_physical_rate_per_spring_N_per_mm,
        spring_physical_rate_per_spring=spring_physical_rate_per_spring,
        spring_physical_model=np.array("direct_linear_spring"),
        spring_motion_ratio=spring_motion_ratio,
        spring_physical_extension=spring_physical_extension,
        spring_physical_extension_hold_curve=spring_physical_extension_hold_curve,
        spring_physical_extension_closed=spring_physical_extension_closed,
        spring_physical_preload_extension=spring_physical_preload_extension,
        spring_initial_tension_per_spring=spring_initial_tension_per_spring,
        spring_force_physical_per_spring=spring_force_physical_per_spring,
        spring_force_physical_hold_curve_per_spring=spring_force_physical_hold_curve_per_spring,
        spring_force_open_physical_per_spring=spring_force_open_physical_per_spring,
        spring_force_closed_physical_per_spring=spring_force_closed_physical_per_spring,
        spring_force_physical_total=spring_force_physical_total,
        spring_energy_stored=spring_energy_stored,
        spring_energy_delta=spring_energy_delta,
        spring_scale_factor=spring_scale_factor,
        F_drive_s_baseline=F_drive_s_baseline,
        F_hold_s_baseline_curve=F_hold_s_baseline_curve,
        F_spring_assist_component=F_spring_assist_component,
        F_spring_anchor=F_spring_anchor,
        F_spring_anchor_norm=np.linalg.norm(F_spring_anchor, axis=1),
        F_spring_physical_anchor=F_spring_physical_anchor,
        F_spring_physical_anchor_norm=np.linalg.norm(F_spring_physical_anchor, axis=1),
        P_spring_storage=P_spring_storage,
    )
    spring_results_written = True
    print("Trekveer-overdekking opgeslagen in:")
    print(spring_path)
    print(f"equiv. veerkracht open / gesloten   : {spring_force_open_total:.2f} N / {spring_force_closed_total:.2f} N per mechanisme")
    print(f"directe veerconstante per veer       : {spring_physical_rate_per_spring_N_per_mm:.3f} N/mm")
    print(f"fysieke veerkracht open/dicht per veer: {spring_force_open_per_spring:.1f} N / {spring_force_closed_per_spring:.1f} N")
    print(f"fysieke veerweg over slag            : {spring_physical_extension_closed*1000:.0f} mm")
    print(f"piekreductie bij openen             : {opening_peak_reduction:.2f} N ({opening_peak_reduction_pct:.1f} %)")
    print(f"max |F_s| met trekveren, openen     : {force_peak_open_spring:.2f} N per mechanisme")
    print(f"max |F_s| met trekveren, sluiten    : {force_peak_close_spring:.2f} N per mechanisme")
    print(f"positieve energie open/sluit        : {E_positive_open_spring:.1f} J / {E_positive_close_spring:.1f} J")
    print(f"max |F_hold| met trekveren          : {np.max(np.abs(F_hold_s_curve_spring)):.2f} N per mechanisme")
else:
    print("Trekveer-case niet berekend.")

if use_spring_assist_for_main_output and spring_results_written:
    recommended_load_case_for_nb4 = "overdekking_trekveren"
else:
    recommended_load_case_for_nb4 = "overdekking"
print(f"Aanbevolen load_case voor Notebook 4: {recommended_load_case_for_nb4}")


## 3D-animatie van de overdekking

De animatie toont dezelfde kinematica als Notebook 1, maar plaatst identieke mechanismen naast elkaar over de breedte. De voorbalk verbindt de K-punten en het doek wordt transparant weergegeven tussen de achterlijn en de voorbalk.


In [ ]:
animation_frame_step = 12
animation_indices = np.arange(0, n_steps, animation_frame_step)
if animation_indices[-1] != n_steps - 1:
    animation_indices = np.append(animation_indices, n_steps - 1)

link_chains = [
    ("3", ["B", "D", "E"]),
    ("4", ["C", "E", "H"]),
    ("5", ["D", "F", "G"]),
    ("6", ["F", "I"]),
    ("7", ["G", "H", "J"]),
    ("8", ["I", "J", "K"]),
]

def points_at(k):
    return {
        "C": C_pos[k], "B": B_pos[k], "D": D_pos[k], "E": E_pos[k], "F": F_pos[k],
        "G": G_pos[k], "H": H_pos[k], "I": I_pos[k], "J": J_pos[k], "K": K_pos[k],
    }

fig3d = plt.figure(figsize=(9, 6))
ax3d = fig3d.add_subplot(111, projection="3d")

def update_3d(frame):
    k = int(animation_indices[frame])
    ax3d.cla()
    ax3d.set_xlim(-0.10, max(0.50, canopy_depth + 0.30))
    ax3d.set_ylim(-canopy_width / 2.0 - 0.30, canopy_width / 2.0 + 0.30)
    ax3d.set_zlim(-L1 - 0.20, 0.35)
    ax3d.set_xlabel("uitval x [m]")
    ax3d.set_ylabel("breedte z [m]")
    ax3d.set_zlabel("hoogte y [m]")
    ax3d.set_title(f"Overdekking t = {t[k]:.2f} s")
    ax3d.view_init(elev=22, azim=-55)

    pts = points_at(k)
    z_min = support_z_positions[0]
    z_max = support_z_positions[-1]
    K_now = pts["K"]
    fabric_vertices = [[
        (0.0, z_min, 0.0),
        (0.0, z_max, 0.0),
        (K_now[0], z_max, K_now[1]),
        (K_now[0], z_min, K_now[1]),
    ]]
    fabric = Poly3DCollection(fabric_vertices, alpha=0.20, facecolor="tab:blue", edgecolor="none")
    ax3d.add_collection3d(fabric)
    ax3d.plot([K_now[0], K_now[0]], [z_min, z_max], [K_now[1], K_now[1]], color="black", linewidth=4, label="voorbalk")
    ax3d.plot([0.0, 0.0], [z_min, z_max], [0.0, 0.0], color="gray", linewidth=3, label="achterlijn")

    for z in support_z_positions:
        ax3d.plot([0.0, 0.0], [z, z], [0.0, -L1], color="black", linewidth=3)
        for _, chain in link_chains:
            xs = [pts[name][0] for name in chain]
            ys = [pts[name][1] for name in chain]
            zs = [z for _ in chain]
            ax3d.plot(xs, zs, ys, "-o", linewidth=2.2, markersize=3.5)
        ax3d.scatter([pts["B"][0]], [z], [pts["B"][1]], color="red", s=35)
    return ax3d

ani3d = FuncAnimation(fig3d, update_3d, frames=len(animation_indices), interval=80, repeat=False)
ani3d_html = ani3d.to_jshtml(default_mode="once")
plt.close(fig3d)
HTML(ani3d_html)
